In [62]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [63]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [64]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [65]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [717]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [718]:
from tqdm import tqdm

def impute_missing_dates(
        df,
        freq='M',
        key=['CHAIN', 'FACILITY_NAME', 'PARENT_MATERIAL_CODE'], 
        date_col='MONTH_DATE',
        max_date='2026-12-31'
    ):

    impute_df = df.copy()
    impute_df[date_col] = pd.to_datetime(impute_df[date_col])
    impute_df['key'] = impute_df[key].astype(str).agg('_'.join, axis=1)
    min_dates_df = impute_df.groupby(
        'key', as_index=False
    )[date_col].min()

    def impute_missing_dates_key(key, min_date, max_date):
        df_imputed = pd.DataFrame(
            pd.date_range(min_date, max_date, freq=freq),
            columns=[date_col]
        )
        df_imputed['key'] = key
        return df_imputed
    
    outputs = Parallel(n_jobs=-1)(
        delayed(impute_missing_dates_key)(row['key'], row[date_col], max_date)
        for idx, row in tqdm(min_dates_df.iterrows())
    )
    df_full = pd.concat(outputs)
    df_full.reset_index(drop=True, inplace=True)

    df_full = df_full.merge(
        impute_df, on=['key', date_col], how='left',
    )

    return df_full

In [719]:
non_rc_to_rc_mapping = {
    718948: 722208,
    718356: 721837,
    718455: 721963,
    718341: 721898,
    718342: 721836,
    719008: 721842,
    729186: 729165,
    718288: 721843,
    722301: 722302
}

rc_to_non_rc_mapping = {value: key for key, value in non_rc_to_rc_mapping.items()}

rc_skus = list(rc_to_non_rc_mapping.keys()) + [729280]
non_rc_skus = list(rc_to_non_rc_mapping.values())

In [720]:
def get_final_psku(x):
    if x['zone'] in ['North', 'East']:
        if (x['rc_psku'] == 1):
            return x['parent_material_code']
        else:
            return non_rc_to_rc_mapping[x['parent_material_code']]
    else:
        if (x['rc_psku'] == 0):
            return x['parent_material_code']
        else:
            return rc_to_non_rc_mapping[x['parent_material_code']]   
        
            
def zepto_saff_golf_fix(df):

    df = df.copy()

    zepto_gold_df = df[
        (df['chain'] == 'Zepto') &
        (df['material_group_code'] == 'SAFF GOLD')
    ]

    rest = df[
        ~((df['chain'] == 'Zepto') &
        (df['material_group_code'] == 'SAFF GOLD'))
    ]

    ########### Offtake City to New City ###########
    city_mappings_df = pd.read_excel(
        r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
        'Final'
    )

    city_mappings_df.rename(columns={'City': 'warehouse_city', 'Final City': 'final'}, inplace=True)

    city_mappings_df.columns = city_mappings_df.columns.str.lower()

    assert city_mappings_df.duplicated(subset=['platform_name', 'city']).sum() == 0

    city_mappings_df.rename(columns={'platform_name': 'chain'}, inplace=True)
    #####################################################################


    #### New City to Zone Mappings #######
    # region_mappings = pd.read_excel(
    #     r"C:\Users\aniket.patel\OneDrive - Marico Ltd\Zepto - SAFF GOLD - Region Mappings.xlsx"    
    # )
    # region_mappings.columns = region_mappings.columns.str.lower()
    data_region = {
        "city": [
            "Faridabad", "Ahmedabad", "Jaipur", "Patiala", "Pune",
            "Bangalore", "GURGAON", "Medak", "Chennai", "Lucknow",
            "Hyderabad", "Jhajjar", "Howrah", "Mumbai", "Bhiwandi"
        ],
        "State/Region": [
            "Haryana", "Gujarat", "Rajasthan", "Punjab", "Maharashtra",
            "Karnataka", "Haryana", "Telangana", "Tamil Nadu", "Uttar Pradesh",
            "Telangana", "Haryana", "West Bengal", "Maharashtra", "Maharashtra"
        ],
        "zone": [
            "North", "West", "North", "North", "West",
            "South", "North", "South", "South", "North",
            "South", "North", "East", "West", "West"
        ]
    }

    region_mappings = pd.DataFrame(data_region)
    #######################################


    ### FIX ####
    len_before_merge = len(zepto_gold_df)
    zepto_gold_df = zepto_gold_df.merge(
        city_mappings_df[['chain', 'city', 'final']],
        on=['chain', 'city'],
        how='left'
    )
    assert len_before_merge == len(zepto_gold_df)
    del len_before_merge

    zepto_gold_df = zepto_gold_df[zepto_gold_df['final'].notna()]

    len_before_merge = len(zepto_gold_df)
    zepto_gold_df = zepto_gold_df.merge(
        region_mappings[['city', 'zone']].rename(
            columns={'city': 'final'}
        ),
        how='left', 
        on=['final']
    )
    assert len_before_merge == len(zepto_gold_df)
    del len_before_merge


    zepto_gold_df['rc_psku'] = zepto_gold_df['parent_material_code'].map(
        lambda x: 1 if x in rc_skus else 0
    )

    zepto_gold_df['final_psku'] = zepto_gold_df.apply(lambda x: get_final_psku(x), axis=1)

    zepto_gold_df = zepto_gold_df.groupby(
        ['chain', 'city', 'final_psku', 'material_group_code', 
        'month_date'], as_index=False
    )['vol_in_roum'].sum()
    zepto_gold_df.rename(columns={'final_psku': 'parent_material_code'}, inplace=True)

    df = pd.concat([rest, zepto_gold_df], ignore_index=True)

    return df

### Primary Actuals + Sec Plan Data

In [721]:
plan_actuals_query = """
-- Primary base table
SELECT 
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    DCM.facility_name,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain IN ('Grofers', 'Swiggy', 'Zepto', 'Kiranakart Technologies')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
LEFT JOIN 
(
    SELECT DISTINCT
        customer,
        facility_name
    FROM
        dev_db.data_science.trn_soh_dc_master
) DCM on MESR.distributor_code = DCM.customer
WHERE month_date > '2023-12-31'
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 4, 3, 5
"""

plan_actuals_df = pd.read_sql(
    plan_actuals_query,
    prod_conn
)

In [722]:
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = pd.to_datetime(plan_actuals_df['month_date'])

In [723]:
plan_actuals_df.duplicated(subset=['chain', 'facility_name', 'parent_material_code', 
     'material_group_code', 'month_date']).sum()

0

In [724]:
plan_actuals_df[plan_actuals_df['facility_name'].isna()]

,chain,facility_name,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
488593,Blinkit,None,718472,ADV-AHO-R,2024-01-01,0.0,0.108491,0.000,0.0
488594,Blinkit,None,718472,ADV-AHO-R,2024-02-01,0.0,0.163414,0.000,0.0
488595,Blinkit,None,718472,ADV-AHO-R,2024-12-31,0.0,0.000000,2.281,0.0
488596,Blinkit,None,718473,ADV-AHO-R,2024-01-01,0.0,0.187036,0.000,0.0
488597,Blinkit,None,718473,ADV-AHO-R,2024-02-01,0.0,0.281743,0.000,0.0
...,...,...,...,...,...,...,...,...,...
966868,Zepto,None,810179,SW_SGPRF,2024-10-01,0.0,11.425066,0.000,0.0
966869,Zepto,None,810179,SW_SGPRF,2024-11-01,0.0,0.466896,0.000,0.0
966870,Zepto,None,810179,SW_SGPRF,2024-12-01,0.0,0.746376,0.000,0.0
966871,Zepto,None,810179,SW_SGPRF,2025-01-01,0.0,0.319030,0.000,0.0


In [725]:
plan_actuals_df.loc[plan_actuals_df['facility_name'].isna(), 'facility_name'] = np.nan

In [726]:
plan_actuals_df['facility_name'] = plan_actuals_df['facility_name'].str.lower()

In [727]:
plan_actuals_df.shape

(966873, 9)

In [728]:
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), column='parent_material_code')

In [729]:
plan_actuals_df

,chain,facility_name,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,ahmedabad a2 - feeder warehouse,718472,ADV-AHO-R,2024-12-01,0.0,0.436871,0.0,0.0
1,Blinkit,ahmedabad a2 - feeder warehouse,718472,ADV-AHO-R,2025-01-01,0.0,0.374589,0.0,0.0
2,Blinkit,ahmedabad a2 - feeder warehouse,718472,ADV-AHO-R,2025-02-01,0.0,0.325482,0.0,0.0
3,Blinkit,ahmedabad a2 - feeder warehouse,718472,ADV-AHO-R,2025-03-01,0.0,0.448389,0.0,0.0
4,Blinkit,ahmedabad a2 - feeder warehouse,718472,ADV-AHO-R,2025-04-01,0.0,0.308430,0.0,0.0
...,...,...,...,...,...,...,...,...,...
966868,Zepto,NaN,810179,SW_SGPRF,2024-10-01,0.0,11.425066,0.0,0.0
966869,Zepto,NaN,810179,SW_SGPRF,2024-11-01,0.0,0.466896,0.0,0.0
966870,Zepto,NaN,810179,SW_SGPRF,2024-12-01,0.0,0.746376,0.0,0.0
966871,Zepto,NaN,810179,SW_SGPRF,2025-01-01,0.0,0.319030,0.0,0.0


In [730]:
plan_actuals_df['month_date'] = plan_actuals_df['month_date'] + MonthEnd(0)

In [731]:
plan_actuals_df.duplicated(subset=['chain', 'facility_name', 'parent_material_code', 
     'material_group_code', 'month_date']).sum()

596169

In [732]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'facility_name', 'parent_material_code', 
     'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [733]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [734]:
plan_actuals_df = plan_actuals_df[plan_actuals_df['material_group_code']!='PABABY_SP']

In [735]:
plan_actuals_df = impute_missing_dates(
    plan_actuals_df.copy(),
    key=['chain', 'facility_name', 'parent_material_code'],
    date_col='month_date'
)

40443it [00:09, 4160.24it/s]


In [736]:
plan_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [737]:
cols = ['chain', 'facility_name', 'parent_material_code', 'material_group_code']

plan_actuals_df[cols] = plan_actuals_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [738]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].fillna(0)

In [739]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [740]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [741]:
plan_actuals_df[plan_actuals_df['facility_name'].isna()]

,month_date,key,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
276634,2024-01-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000409,0.0,0.0
276635,2024-02-29,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000410,0.0,0.0
276636,2024-03-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
276637,2024-04-30,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
276638,2024-05-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
944931,2026-08-31,Zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,0.0,0.000000,0.0,0.0
944932,2026-09-30,Zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,0.0,0.000000,0.0,0.0
944933,2026-10-31,Zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,0.0,0.000000,0.0,0.0
944934,2026-11-30,Zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,0.0,0.000000,0.0,0.0


In [742]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [743]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].clip(lower=0)

In [744]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

In [745]:
plan_actuals_df

,month_date,key,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,2026-06-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.021845,0.0,0.0
1,2026-07-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0
2,2026-08-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0
3,2026-09-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0
4,2026-10-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
962180,2026-08-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0
962181,2026-09-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0
962182,2026-10-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0
962183,2026-11-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0


In [746]:
plan_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [747]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [748]:
plan_actuals_df['Primary P3M'] = plan_actuals_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [749]:
plan_actuals_df.sample(1)

,month_date,key,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
927723,2026-07-31,Zepto_mum-dry-mh3_721133,Zepto,mum-dry-mh3,721133,PA_JASGLD,0.0,0.0,0.0,0.0,9.6


In [750]:
plan_actuals_df

,month_date,key,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2026-06-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.021845,0.0,0.0,NaN
1,2026-07-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
2,2026-08-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
3,2026-09-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
4,2026-10-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
962180,2026-08-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962181,2026-09-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962182,2026-10-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962183,2026-11-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0


In [751]:
plan_actuals_df.groupby(
    ['chain', 'parent_material_code', 
     'material_group_code', 'month_date'], as_index=False, dropna=False
).sum().to_csv('plan_actuals_aggregated.csv', index=False)

In [752]:
plan_actuals_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

### Offtakes

In [753]:
offtakes_monthly_query = """
SELECT
    OTM.platform_name AS chain,
    OTM.city,
    MM.parent_material_code,
    MM.material_group_code,
    OTM.month_date,
    SUM(CASE
        WHEN MM.uom_reporting IN ('KL', 'TO') THEN ROUND(OTM.vol_in_lit / POW(10, 3), 4)
        ELSE OTM.vol_in_lit
    END) AS vol_in_roum
FROM 
    dwh_ecommplatform_offtake OTM
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON OTM.material_code = MM.material_code
WHERE
    OTM.platform_name IN ('Blinkit', 'Swiggy', 'Zepto') AND
    OTM.month_date > '2022-12-31' 
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 3, 5, 4
""" 

offtakes_monthly_df = pd.read_sql(
    offtakes_monthly_query,
    prod_conn
)

In [754]:
offtakes_monthly_df.columns = offtakes_monthly_df.columns.str.lower()
offtakes_monthly_df['month_date'] = pd.to_datetime(offtakes_monthly_df['month_date'])
offtakes_monthly_df.head()

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum
0,Blinkit,Agra,718312,PCNO(R),2024-08-31,0.003
1,Blinkit,Agra,718312,PCNO(R),2024-09-30,0.012
2,Blinkit,Agra,718312,PCNO(R),2024-10-31,0.024
3,Blinkit,Agra,718312,PCNO(R),2024-11-30,0.031
4,Blinkit,Agra,718312,PCNO(R),2024-12-31,0.040


In [755]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-04-30 00:00:00')

In [756]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [757]:
offtakes_monthly_df = zepto_saff_golf_fix(offtakes_monthly_df.copy())

In [758]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [759]:
offtakes_monthly_df[offtakes_monthly_df['run_month'] != offtakes_monthly_df['month_date']]

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month


In [760]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

597600

In [761]:
offtakes_monthly_df = realign_pskus(offtakes_monthly_df.copy(), 'parent_material_code')

In [762]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

597600

In [763]:
offtakes_monthly_df = offtakes_monthly_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 
     'month_date', 'run_month'], as_index=False
)['vol_in_roum'].sum()

In [764]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-04-30 00:00:00')

#### Add m month forecast

In [765]:
mmonth_ot_df = pd.read_sql("""
SELECT * FROM TRN_DF_QCOM_OFFTAKE_CHAIN_PSKU
WHERE run_month='2026-06-30' AND month_date='2026-05-31'  
""",
dev_conn
)

In [766]:
mmonth_ot_df

,MONTH_DATE,KEY,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,RUN_MONTH,IMPUTED
0,2026-05-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,64.1280,2026-06-30,0
1,2026-05-31,blinkit_718310,blinkit,718310.0,PCNO(R),0.0000,2026-06-30,0
2,2026-05-31,blinkit_718312,blinkit,718312.0,PCNO(R),13.3020,2026-06-30,0
3,2026-05-31,blinkit_718315,blinkit,718315.0,PCNO(R),0.0000,2026-06-30,0
4,2026-05-31,blinkit_718317,blinkit,718317.0,H&C,0.0000,2026-06-30,0
...,...,...,...,...,...,...,...,...
873,2026-05-31,zepto_810685,zepto,810685.0,SAF-MUSLI,0.0016,2026-06-30,0
874,2026-05-31,zepto_810738,zepto,810738.0,PABABY_GM,48.5080,2026-06-30,0
875,2026-05-31,zepto_810971,zepto,810971.0,PA_ESS_HO,0.6300,2026-06-30,0
876,2026-05-31,zepto_811005,zepto,811005.0,PA_ESS_HO,0.7140,2026-06-30,0


In [767]:
mmonth_ot_df.columns = mmonth_ot_df.columns.str.lower()
mmonth_ot_df.rename(
    columns={
        'platform_name': 'chain',
        'brand_code': 'material_group_code',
        'vol_in_rum': 'vol_in_roum'
    }, 
    inplace=True
)

In [768]:
mmonth_ot_df['parent_material_code'] = mmonth_ot_df['parent_material_code'].astype(int)

In [769]:
mmonth_ot_df['month_date'] = pd.to_datetime(mmonth_ot_df['month_date'])
mmonth_ot_df['run_month'] = pd.to_datetime(mmonth_ot_df['run_month'])

####

In [770]:
mmonth_ot_df['key'] = mmonth_ot_df['key'].str.capitalize()
mmonth_ot_df['chain'] = mmonth_ot_df['chain'].str.capitalize()
mmonth_ot_df

,month_date,key,chain,parent_material_code,material_group_code,vol_in_roum,run_month,imputed
0,2026-05-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,64.1280,2026-06-30,0
1,2026-05-31,Blinkit_718310,Blinkit,718310,PCNO(R),0.0000,2026-06-30,0
2,2026-05-31,Blinkit_718312,Blinkit,718312,PCNO(R),13.3020,2026-06-30,0
3,2026-05-31,Blinkit_718315,Blinkit,718315,PCNO(R),0.0000,2026-06-30,0
4,2026-05-31,Blinkit_718317,Blinkit,718317,H&C,0.0000,2026-06-30,0
...,...,...,...,...,...,...,...,...
873,2026-05-31,Zepto_810685,Zepto,810685,SAF-MUSLI,0.0016,2026-06-30,0
874,2026-05-31,Zepto_810738,Zepto,810738,PABABY_GM,48.5080,2026-06-30,0
875,2026-05-31,Zepto_810971,Zepto,810971,PA_ESS_HO,0.6300,2026-06-30,0
876,2026-05-31,Zepto_811005,Zepto,811005,PA_ESS_HO,0.7140,2026-06-30,0


In [771]:
offtakes_monthly_df = pd.concat([offtakes_monthly_df, mmonth_ot_df], ignore_index=True)

In [772]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [773]:
offtakes_monthly_df = impute_missing_dates(
    offtakes_monthly_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

890it [00:00, 3537.80it/s]


In [774]:
offtakes_monthly_df.sort_values(by=['key', 'month_date'], inplace=True)

cols = ['chain', 'parent_material_code', 'material_group_code']

offtakes_monthly_df[cols] = offtakes_monthly_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [775]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [776]:
offtakes_monthly_df['vol_in_roum'] = offtakes_monthly_df['vol_in_roum'].fillna(0)

In [777]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [778]:
(offtakes_monthly_df['key'] == offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [779]:
offtakes_monthly_df.rename(columns={'vol_in_roum': 'offtake_vol_rum'}, inplace=True)

In [780]:
offtakes_monthly_df.groupby(['month_date'])['offtake_vol_rum'].sum()

month_date
2023-01-31     24974.523800
2023-02-28     21910.917700
2023-03-31     22741.199600
2023-04-30     18869.489800
2023-05-31     18075.762900
2023-06-30     21990.413600
2023-07-31     24142.810900
2023-08-31     23501.260600
2023-09-30     17823.328300
2023-10-31     21403.203600
2023-11-30     23992.043500
2023-12-31     24176.780200
2024-01-31     26834.176400
2024-02-29     21851.471500
2024-03-31     25431.763200
2024-04-30     22633.553800
2024-05-31     19660.621700
2024-06-30     21509.297100
2024-07-31     22640.069900
2024-08-31     24427.248600
2024-09-30     26332.478700
2024-10-31     33481.273400
2024-11-30     50496.037100
2024-12-31     59153.637300
2025-01-31     63434.042400
2025-02-28     69783.569000
2025-03-31     77069.482500
2025-04-30     70404.555900
2025-05-31     67573.361400
2025-06-30     69729.655600
2025-07-31     76385.507700
2025-08-31     75733.195200
2025-09-30     78360.921000
2025-10-31    100520.035700
2025-11-30    161379.758700
2025-12-3

In [781]:
# offtakes_monthly_df[offtakes_monthly_df['month_date'] < '2026-02-28'].to_excel('Offtakes Chain PSKU till Jan26.xlsx', index=False)

In [782]:
offtakes_monthly_df


,month_date,key,chain,parent_material_code,material_group_code,run_month,offtake_vol_rum,imputed
0,2023-01-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-01-31,22.584,NaN
1,2023-02-28,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-02-28,17.568,NaN
2,2023-03-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-03-31,25.332,NaN
3,2023-04-30,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-04-30,21.888,NaN
4,2023-05-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-05-31,15.860,NaN
...,...,...,...,...,...,...,...,...
29554,2026-08-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-08-31,0.000,NaN
29555,2026-09-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-09-30,0.000,NaN
29556,2026-10-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-10-31,0.000,NaN
29557,2026-11-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-11-30,0.000,NaN


### SOH

In [783]:
soh_query = """
SELECT 
    SOH.final_chains_ho AS chain,
    DCM.facility_name,
    SOH.parent_sku AS parent_material_code,
    SOH.as_on_date,
    SUM(current_soh) AS current_soh
FROM dev_db.public.dwh_soh_oos SOH
LEFT JOIN 
(
    SELECT DISTINCT
        customer,
        facility_name
    FROM
        dev_db.data_science.trn_soh_dc_master
) DCM on SOH.distributorcode = DCM.customer
WHERE 
    final_chains_ho IN ('Swiggy', 'Blinkit', 'Zepto') AND
    vol_bpm='Vol'
GROUP BY
    1, 2, 3, 4
ORDER BY
    1, 2, 3, 4
"""

soh_df = pd.read_sql(
    soh_query,
    dev_conn
)

In [784]:
soh_df.columns = soh_df.columns.str.lower()

In [785]:
soh_df.duplicated(subset=['chain', 'facility_name', 'parent_material_code', 'as_on_date']).sum()

0

In [786]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])
# soh_df[soh_df['as_on_date'] >= '2026-01-31'].to_csv('soh_recent_qcom.csv', index=False)

In [787]:
soh_df = realign_pskus(soh_df.copy(), 'parent_material_code')

In [788]:
soh_df.duplicated(subset=['chain', 'facility_name', 'parent_material_code', 'as_on_date']).sum()

115524

In [789]:
soh_df = soh_df.groupby(
    ['chain', 'facility_name', 'parent_material_code', 'as_on_date'],
    as_index=False
)['current_soh'].sum()

In [790]:
soh_df['as_on_date'].max()

Timestamp('2026-06-02 00:00:00')

In [791]:
soh_df['current_soh'].min()

0.0

In [792]:
soh_df.duplicated(subset=['chain', 'facility_name', 'parent_material_code', 'as_on_date']).sum()

0

In [793]:
soh_df['facility_name'] = soh_df['facility_name'].str.lower()

In [794]:
soh_df['key2'] = soh_df[['chain', 'facility_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [795]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])

In [796]:
soh_df['run_month'] = soh_df['as_on_date'] + MonthEnd(0)

In [797]:
date_wise_soh_vol_sum = soh_df.groupby(['as_on_date'], as_index=False)['current_soh'].sum()

In [798]:
date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]

,as_on_date,current_soh
0,2025-01-19,0.0
1,2025-01-20,0.0
19,2025-07-31,0.0
22,2025-08-03,0.0
29,2025-08-12,0.0
34,2025-08-20,0.0
40,2025-08-28,0.0
53,2025-09-15,0.0
56,2025-09-20,0.0
60,2025-09-25,0.0


In [799]:
soh_df[
    soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]['current_soh'].sum()

0.0

In [800]:
soh_df = soh_df[
    ~soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]

In [801]:
soh_df['current_soh'].sum()

14486172.605135368

In [802]:
soh_df.groupby(['chain'])['as_on_date'].max()

chain
Blinkit   2026-06-02
Swiggy    2026-06-02
Zepto     2026-06-02
Name: as_on_date, dtype: datetime64[ns]

# Remove the below cell

In [803]:
### THIS IS TEMP FILTER ###
soh_df = soh_df[soh_df['as_on_date'] < '2026-06-01']

### Forecasts

In [804]:
# forecasts = pd.read_excel(
#     r"/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_chain_psku_june_live.xlsx",
#     sheet_name='Base'
# )
# forecasts = forecasts[forecasts['key'].notna()]
# forecasts.head()

In [805]:
# xx = forecasts.copy()
forecasts = xx.copy()

In [806]:
# forecasts['final_heuristic_60_prophet_value_2'].sum()

In [807]:
forecasts.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf',
       ...
       'P3M_non_seasonal_value', 'P6M_non_seasonal_value',
       'LY_P3M_non_seasonal', 'LY_P6M_non_seasonal',
       'LY_P3M_non_seasonal_value', 'LY_P6M_non_seasonal_value',
       'non_seasonal_growth', 'final_seasonal_month', 'final_heuristic_value',
       'month_different'],
      dtype='object', length=135)

In [808]:
# Final Model Output

In [809]:
# forecasts.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
forecasts['Final Heuristic Prophet Vol'] = forecasts['final_heuristic_value'] * (10 ** 7) / forecasts['qtr_ind_rate']

In [810]:
forecasts.columns.to_list()

['key',
 'month_date',
 'pred_p3m',
 'pred_p6m',
 'pred_prophet',
 'pred_rf',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'parent_material_code',
 'platform_name',
 'brand_code',
 'qtr_ind_rate',
 'vol_in_rum_value',
 'pred_best_model',
 'pred_value_best_model',
 'vol_in_rum_treated',
 'vol_in_rum_value_treated',
 'train_till',
 'cov',
 'run',
 'step',
 'file_path',
 'run_month_x',
 'M month',
 'portfolio',
 'pred_prophet_70%ile',
 'vol_in_rum',
 'P3M',
 'P6M',
 'LY P3M',
 'LY P6M',
 'LY P3M_copy',
 'P3M Max',
 'P3M Top 2 Mean',
 'MoM P3M growth',
 'MoM P3M growth_lag_1',
 'MoM P3M growth_lag_2',
 '>=20%_3M_inc_month_count',
 'Avg(P3M Mean, Max)',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'LY',
 'LLY',
 'LY value',
 'LLY value',
 'OT_Value_in_Cr_lag_1',
 'OT_Value_in_Cr_lag_2',
 'OT_Value_in_Cr_lag_3',
 'class',
 'skipped',
 'seasonality_flag',
 'final_trend',
 'lower_threshold',
 'upper_threshold',


In [811]:
forecasts['parent_material_code'] = forecasts['parent_material_code'].astype(int)

In [812]:
forecasts['key'] = forecasts[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [813]:
# forecasts['Final Heuristic Prophet Vol'] = np.where(
#     forecasts['skipped'] == 1,
#     np.where(
#         (forecasts['Potential Seasonal Brand'] == 1) &
#         (forecasts['LY'].notna()),
#         forecasts['LY'],
#         forecasts['P3M']
#     ),
#     forecasts['Final Heuristic Prophet Vol']
# )

In [814]:
# forecasts[forecasts['Final Heuristic Prophet Vol'] != forecasts['final_heuristic_60_prophet_vol_2']][['Final Heuristic Prophet Vol', 'final_heuristic_60_prophet_vol_2']]

In [815]:
forecasts.rename(columns = {'run_month_x': 'run_month'}, inplace=True)

In [816]:
forecasts = forecasts[['key', 'month_date', 'platform_name', 'parent_material_code', 'brand_code', 
                       'run_month', 'M month', 'Final Heuristic Prophet Vol', 'skipped']]

In [817]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [818]:
# forecasts = forecasts[forecasts['run_month'].isin(
#     ['2025-08-31', '2025-09-30']
# )]

In [819]:
forecasts['key'] = forecasts['key'].str.capitalize()
forecasts['platform_name'] = forecasts['platform_name'].str.capitalize()
forecasts


,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,Blinkit_718288,2026-06-30,Blinkit,718288,SAFF GOLD,2026-06-30,M,64.857451,0
1,Blinkit_718288,2026-07-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+1,67.978393,0
2,Blinkit_718288,2026-08-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+2,65.531517,0
3,Blinkit_718288,2026-09-30,Blinkit,718288,SAFF GOLD,2026-06-30,M+3,63.868398,0
4,Blinkit_718288,2026-10-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+4,69.072052,0
...,...,...,...,...,...,...,...,...,...
7211,Zepto_811181,2026-10-31,Zepto,811181,SAF_CDPRS,2026-06-30,M+4,0.203667,1
7212,Zepto_811181,2026-11-30,Zepto,811181,SAF_CDPRS,2026-06-30,M+5,0.203667,1
7213,Zepto_811181,2026-12-31,Zepto,811181,SAF_CDPRS,2026-06-30,M+6,0.203667,1
7214,Zepto_811181,2027-01-31,Zepto,811181,SAF_CDPRS,2026-06-30,M+7,0.203667,1


In [820]:
forecasts.duplicated(['key', 'run_month', 'month_date']).sum()

0

In [821]:
forecasts[forecasts.duplicated(['key', 'run_month', 'month_date'])]

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped


In [822]:
forecasts.rename(columns={'final_vol': 'Final Heuristic Prophet Vol'}, inplace=True)

### Collate Everything

In [823]:
print(f"SOH", soh_df['chain'].unique())
print("Offtakes:", offtakes_monthly_df['chain'].unique())
print("Plan Actuals:", plan_actuals_df['chain'].unique())

SOH ['Blinkit' 'Swiggy' 'Zepto']
Offtakes: ['Blinkit' 'Swiggy' 'Zepto']
Plan Actuals: ['Blinkit' 'Swiggy' 'Zepto']


In [824]:
plan_actuals_df.rename(columns={'key': 'key2'}, inplace=True)

In [825]:
final_df = plan_actuals_df[
    ['key2', 'chain', 'facility_name', 'parent_material_code', 'material_group_code']
].drop_duplicates()

tmp_df = pd.DataFrame()

for rm in ['2026-06-30']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_df.copy()
        tmp_df2['run_month'] = pd.to_datetime(rm)
        tmp_df2['month_date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_df = tmp_df.copy()    
del tmp_df

### Merge Plan

In [826]:
plan_actuals_df.duplicated(subset=['key2', 'month_date']).sum()

0

In [827]:
plan_actuals_df.head()

,month_date,key2,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2026-06-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.021845,0.0,0.0,NaN
1,2026-07-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
2,2026-08-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
3,2026-09-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
4,2026-10-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0


In [828]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    plan_actuals_df[['key2', 'month_date', 'pri_actuals_vol_rum', 'sec_apo_plan_vol_rum', 'Primary P3M']],
    on=['key2', 'month_date'],
    how='left'
)
assert len(final_df) == len_before_merge
del len_before_merge

In [829]:
plan_actuals_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [830]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD,2026-06-30,2026-05-31,2.352,6.4934,7.432
2,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000
3,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000
4,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000
...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811021,Zepto,pun-dry-mh2-koregaon,811021,PADV_WIPS,2026-06-30,2027-02-28,NaN,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811169,Zepto,pun-dry-mh2-koregaon,811169,SW_SGPRF,2026-06-30,2027-02-28,NaN,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811267,Zepto,pun-dry-mh2-koregaon,811267,PA_ESS_HO,2026-06-30,2027-02-28,NaN,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811269,Zepto,pun-dry-mh2-koregaon,811269,PA_ESS_HO,2026-06-30,2027-02-28,NaN,NaN,NaN


In [831]:
final_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
0,2026-06-30,2026-05-31
40443,2026-06-30,2026-06-30
80886,2026-06-30,2026-07-31
121329,2026-06-30,2026-08-31
161772,2026-06-30,2026-09-30
202215,2026-06-30,2026-10-31
242658,2026-06-30,2026-11-30
283101,2026-06-30,2026-12-31
323544,2026-06-30,2027-01-31
363987,2026-06-30,2027-02-28


In [832]:
mmonth_df = final_df[
    final_df['month_date'] < final_df['run_month']
]

final_df = final_df[
    final_df['month_date'] >= final_df['run_month']
]

final_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
40443,2026-06-30,2026-06-30
80886,2026-06-30,2026-07-31
121329,2026-06-30,2026-08-31
161772,2026-06-30,2026-09-30
202215,2026-06-30,2026-10-31
242658,2026-06-30,2026-11-30
283101,2026-06-30,2026-12-31
323544,2026-06-30,2027-01-31
363987,2026-06-30,2027-02-28


In [833]:
mmonth_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
0,2026-06-30,2026-05-31


In [834]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [835]:
for col in ['Primary P3M']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key2'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [836]:
final_df = pd.concat([mmonth_df, final_df], ignore_index=True)

### Merge offtakes

In [837]:
final_df['key'] = final_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [838]:
final_df.duplicated(subset=['run_month', 'month_date', 'key2']).sum()

0

In [839]:
### Monthly Actual Offtakes
len_before_merge = len(final_df)
final_df = final_df.merge(
    offtakes_monthly_df[['month_date', 'key', 'offtake_vol_rum']].rename(columns={
        'offtake_vol_rum': 'Offtake Chain PSKU'
    }),
    on=['month_date', 'key'],
    how='left'
)   
assert len_before_merge == len(final_df)
del len_before_merge

In [840]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,key,Offtake Chain PSKU
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,Blinkit_718287,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD,2026-06-30,2026-05-31,2.352,6.4934,7.432,Blinkit_718288,64.128
2,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718299,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718308,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718310,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.000,0.0000,0.000,Zepto_811279,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.000,0.0000,0.000,Zepto_811279,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.000,0.0000,0.000,Zepto_811279,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.000,Zepto_811279,NaN


In [841]:
final_df['Primary Actuals Chain PSKU P3M Sum Vol'] = final_df.groupby(
    ['run_month', 'chain', 'parent_material_code', 'month_date'], as_index=False, group_keys=False
)['Primary P3M'].transform('sum')

In [842]:
final_df['Group Size'] = final_df.groupby(
    ['run_month', 'chain', 'parent_material_code', 'month_date'], as_index=False, group_keys=False
)['Primary P3M'].transform('size')

In [843]:
final_df['Fallback Contribution'] = 1 / final_df['Group Size']

In [844]:
final_df['Contribution'] = final_df['Primary P3M'].fillna(0) / final_df['Primary Actuals Chain PSKU P3M Sum Vol']

In [845]:
final_df['Final Contribution'] = np.where(
    final_df['Contribution'].isna(),
    final_df['Fallback Contribution'],
    final_df['Contribution']
)

In [846]:
final_df.groupby(
    ['key', 'run_month', 'month_date']
)['Final Contribution'].sum().max()

1.0000000000000002

In [847]:
final_df['Offtake Chain FC PSKU'] = final_df['Final Contribution'] * final_df['Offtake Chain PSKU']

In [848]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [849]:
final_df['M month'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(x['month_date'], np.nan),
    axis=1
)

In [850]:
final_df[['run_month', 'month_date', 'M month']].drop_duplicates()

,run_month,month_date,M month
0,2026-06-30,2026-05-31,NaN
40443,2026-06-30,2026-06-30,M
40444,2026-06-30,2026-07-31,M+1
40445,2026-06-30,2026-08-31,M+2
40446,2026-06-30,2026-09-30,M+3
40447,2026-06-30,2026-10-31,M+4
40448,2026-06-30,2026-11-30,M+5
40449,2026-06-30,2026-12-31,M+6
40450,2026-06-30,2027-01-31,M+7
40451,2026-06-30,2027-02-28,M+8


### Merge Forecasts

In [851]:
forecasts.head()

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,Blinkit_718288,2026-06-30,Blinkit,718288,SAFF GOLD,2026-06-30,M,64.857451,0
1,Blinkit_718288,2026-07-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+1,67.978393,0
2,Blinkit_718288,2026-08-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+2,65.531517,0
3,Blinkit_718288,2026-09-30,Blinkit,718288,SAFF GOLD,2026-06-30,M+3,63.868398,0
4,Blinkit_718288,2026-10-31,Blinkit,718288,SAFF GOLD,2026-06-30,M+4,69.072052,0


In [852]:
forecasts.duplicated(subset=['key', 'month_date', 'run_month']).sum()

0

In [853]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    forecasts[['key', 'month_date', 'run_month', 'Final Heuristic Prophet Vol']],
    on=['key', 'month_date', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [854]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,key,Offtake Chain PSKU,Primary Actuals Chain PSKU P3M Sum Vol,Group Size,Fallback Contribution,Contribution,Final Contribution,Offtake Chain FC PSKU,M month,Final Heuristic Prophet Vol
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,Blinkit_718287,NaN,0.000,44,0.022727,NaN,0.022727,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD,2026-06-30,2026-05-31,2.352,6.4934,7.432,Blinkit_718288,64.128,71.954,43,0.023256,0.103288,0.103288,6.623666,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718299,NaN,0.000,49,0.020408,NaN,0.020408,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718308,NaN,0.000,37,0.027027,NaN,0.027027,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO(R),2026-06-30,2026-05-31,0.000,0.0000,0.000,Blinkit_718310,0.000,0.000,45,0.022222,NaN,0.022222,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.000,0.0000,0.000,Zepto_811279,NaN,0.000,20,0.050000,NaN,0.050000,NaN,M+4,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.000,0.0000,0.000,Zepto_811279,NaN,0.000,20,0.050000,NaN,0.050000,NaN,M+5,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.000,0.0000,0.000,Zepto_811279,NaN,0.000,20,0.050000,NaN,0.050000,NaN,M+6,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.000,Zepto_811279,NaN,0.000,20,0.050000,NaN,0.050000,NaN,M+7,NaN


In [855]:
final_df.rename(columns={'Final Heuristic Prophet Vol': 'Offtake Chain PSKU Forecast Vol'}, inplace=True)

In [856]:
final_df['Offtake Chain FC PSKU Forecast Vol'] = final_df['Final Contribution'] * final_df['Offtake Chain PSKU Forecast Vol']

### Norms

In [857]:
norm_days = pd.read_csv(r"/data/aman_singh/acuuracy_check/Norms 202606_cpsku.csv")

In [858]:
norms = final_df.copy()

In [859]:
norms = norms[['key2', 'run_month', 'month_date', 'Offtake Chain FC PSKU Forecast Vol']]

In [860]:
norms['total_days_in_month'] = norms['month_date'].dt.day

In [861]:
len_before_merge = len(norms)
norms = norms.merge(
    norm_days[['key2', 'norm_days']],
    on=['key2'],
    how='left'
)
assert len_before_merge == len(norms)
del len_before_merge

In [862]:
norms['norm_days'] = norms['norm_days'].fillna(5)

In [863]:
norms['safety_stock'] = norms['Offtake Chain FC PSKU Forecast Vol'] *  norms['norm_days'] / norms['total_days_in_month']

In [864]:
norms.isna().sum()

key2                                       0
run_month                                  0
month_date                                 0
Offtake Chain FC PSKU Forecast Vol    222954
total_days_in_month                        0
norm_days                                  0
safety_stock                          222954
dtype: int64

In [865]:
norms

,key2,run_month,month_date,Offtake Chain FC PSKU Forecast Vol,total_days_in_month,norm_days,safety_stock
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,2026-06-30,2026-05-31,NaN,31,5.000000,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718288,2026-06-30,2026-05-31,NaN,31,5.449809,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718299,2026-06-30,2026-05-31,NaN,31,5.000000,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718308,2026-06-30,2026-05-31,NaN,31,5.000000,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718310,2026-06-30,2026-05-31,NaN,31,5.000000,NaN
...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-10-31,NaN,31,5.000000,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-11-30,NaN,30,5.000000,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-12-31,NaN,31,5.000000,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2027-01-31,NaN,31,5.000000,NaN


In [866]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain FC PSKU Forecast Vol', 'total_days_in_month', 'norm_days'], axis=1).rename(columns={
        'safety_stock': 'norms_soh'
    }),
    on=['key2', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [867]:
norms['month_date'] = norms['month_date'] - MonthEnd(1)

In [868]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain FC PSKU Forecast Vol', 'total_days_in_month'], axis=1),
    on=['key2', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [869]:
final_df.sort_values(
    by=['run_month', 'key2', 'month_date'], inplace=True
)

In [870]:
final_df['safety_stock'] = final_df['safety_stock'].fillna(0)

In [871]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [872]:
norms

,key2,run_month,month_date,Offtake Chain FC PSKU Forecast Vol,total_days_in_month,norm_days,safety_stock
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,2026-06-30,2026-04-30,NaN,31,5.000000,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718288,2026-06-30,2026-04-30,NaN,31,5.449809,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718299,2026-06-30,2026-04-30,NaN,31,5.000000,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718308,2026-06-30,2026-04-30,NaN,31,5.000000,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718310,2026-06-30,2026-04-30,NaN,31,5.000000,NaN
...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-09-30,NaN,31,5.000000,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-10-31,NaN,30,5.000000,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-11-30,NaN,31,5.000000,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,2026-06-30,2026-12-31,NaN,31,5.000000,NaN


In [873]:
chain_wise_max_soh_dates = soh_df.groupby(
    ['run_month', 'chain'], as_index=False
)['as_on_date'].max()

chain_wise_max_soh_dates

,run_month,chain,as_on_date
0,2025-07-31,Blinkit,2025-07-30
1,2025-07-31,Swiggy,2025-07-30
2,2025-07-31,Zepto,2025-07-30
3,2025-08-31,Blinkit,2025-08-30
4,2025-08-31,Swiggy,2025-08-30
5,2025-08-31,Zepto,2025-08-30
6,2025-09-30,Blinkit,2025-09-30
7,2025-09-30,Swiggy,2025-09-30
8,2025-09-30,Zepto,2025-09-30
9,2025-10-31,Blinkit,2025-10-31


In [874]:
chain_wise_max_soh_dates = (
    chain_wise_max_soh_dates
    .groupby('chain')
    .apply(lambda x: dict(zip(x['run_month'], x['as_on_date'])))
    .to_dict()
)

In [875]:
last_date_soh = pd.DataFrame()

for c in chain_wise_max_soh_dates.keys():
    for rm in chain_wise_max_soh_dates[c].keys():
        last_date_soh = pd.concat([
            last_date_soh,
            soh_df[
                (soh_df['chain'] == c) &
                (soh_df['as_on_date'] == chain_wise_max_soh_dates[c][rm])
            ]
        ])

In [876]:
last_date_soh['as_on_date'] = last_date_soh['as_on_date'] + MonthEnd(0)

In [877]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    last_date_soh[['key2', 'as_on_date', 'current_soh']].rename(
        columns={
            'as_on_date': 'month_date',
            'current_soh': 'Actual Closing SOH'
        }
    ),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [878]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [879]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,Contribution,Final Contribution,Offtake Chain FC PSKU,M month,Offtake Chain PSKU Forecast Vol,Offtake Chain FC PSKU Forecast Vol,norms_soh,norm_days,safety_stock,Actual Closing SOH
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,0.022727,NaN,NaN,NaN,NaN,NaN,5.0,0.0,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,0.022727,NaN,M,NaN,NaN,NaN,5.0,0.0,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,0.022727,NaN,M+1,NaN,NaN,NaN,5.0,0.0,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,0.022727,NaN,M+2,NaN,NaN,NaN,5.0,0.0,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,0.022727,NaN,M+3,NaN,NaN,NaN,5.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,0.050000,NaN,M+4,NaN,NaN,NaN,5.0,0.0,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,0.050000,NaN,M+5,NaN,NaN,NaN,5.0,0.0,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,0.050000,NaN,M+6,NaN,NaN,NaN,5.0,0.0,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,0.050000,NaN,M+7,NaN,NaN,NaN,5.0,0.0,NaN


In [880]:
final_df['Actual Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key2']
)['Actual Closing SOH'].shift(1)


final_df['Actual Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key2']
)['Actual Closing SOH'].shift(2)

In [881]:
final_df['safety_stock'].sum()

318535.9153404502

In [882]:
final_df['safety_stock'].min()

0.0

In [883]:
# final_df['Assumed Closing SOH'] = np.where(
#     final_df['M month'] == 'M',  
#     final_df['Actual Closing SOH_Lag_1'].fillna(0) + final_df['sec_apo_plan_vol_rum'].fillna(0) \
#     - final_df['Offtake Chain FC PSKU Forecast Vol'].fillna(0),
#     final_df['safety_stock']
# )

final_df['Assumed Closing SOH'] = final_df['safety_stock']

In [884]:
final_df['Assumed Closing SOH'] = final_df['Assumed Closing SOH'].clip(lower=0.0)

In [885]:
final_df['Assumed Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key2']
)['Assumed Closing SOH'].shift(1)

final_df['Assumed Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key2']
)['Assumed Closing SOH'].shift(2)

In [886]:
final_df = final_df[
    ~final_df['material_group_code'].isin(['NC FREE', 'HC FREE'])
]

In [887]:
soh_df['as_on_date'].max()

Timestamp('2026-05-30 00:00:00')

### Add P3M, LY

In [888]:
plan_actuals_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [889]:
actuals_df = plan_actuals_df.copy()
actuals_df['month_date'] = actuals_df['month_date'] + MonthEnd(0)

In [890]:
actuals_df = actuals_df.groupby(
    ['key2', 'month_date'], as_index=False
)[['pri_actuals_vol_rum', 'sec_actuals_vol_rum']].sum()

In [891]:
actuals_df.duplicated(subset=['key2', 'month_date']).sum()

0

In [892]:
actuals_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [893]:
actuals_df.sort_values(by=['key2', 'month_date'], inplace=True)

In [894]:
actuals_df['Primary P3M redundant'] = actuals_df.groupby(
['key2'], as_index = False, group_keys = False)['pri_actuals_vol_rum'].shift(1)\
                            .rolling(window=3, min_periods=3).mean()

In [895]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'Primary Actuals Vol',
        'sec_actuals_vol_rum': 'Sec Actuals Vol'
    }),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [896]:
final_df.isna().sum()

key2                                           0
chain                                          0
facility_name                               9580
parent_material_code                           0
material_group_code                            0
run_month                                      0
month_date                                     0
pri_actuals_vol_rum                        81206
sec_apo_plan_vol_rum                       81206
Primary P3M                                  329
key                                            0
Offtake Chain PSKU                        226062
Primary Actuals Chain PSKU P3M Sum Vol         0
Group Size                                     0
Fallback Contribution                          0
Contribution                              235456
Final Contribution                             0
Offtake Chain FC PSKU                     226062
M month                                    40443
Offtake Chain PSKU Forecast Vol           222954
Offtake Chain FC PSK

In [897]:
plan_actuals_df

,month_date,key2,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2026-06-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.021845,0.0,0.0,NaN
1,2026-07-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
2,2026-08-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
3,2026-09-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
4,2026-10-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
962180,2026-08-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962181,2026-09-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962182,2026-10-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962183,2026-11-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0


In [898]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,safety_stock,Actual Closing SOH,Actual Closing SOH_Lag_1,Actual Closing SOH Lag 2,Assumed Closing SOH,Assumed Closing SOH_Lag_1,Assumed Closing SOH Lag 2,Primary Actuals Vol,Sec Actuals Vol,Primary P3M redundant
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN


In [899]:
ly_actuals_df = actuals_df.copy()

In [900]:
ly_actuals_df['month_date'] = ly_actuals_df['month_date'] + MonthEnd(12)

In [901]:
ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M'
    })

,key2,month_date,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,2027-06-30,0.0,0.0,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,2027-07-31,0.0,0.0,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,2027-08-31,0.0,0.0,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,2027-09-30,0.0,0.0,0.0
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,2027-10-31,0.0,0.0,0.0
...,...,...,...,...,...
962180,Zepto_pun-dry-mh2-koregaon_811279,2027-08-31,0.0,0.0,0.0
962181,Zepto_pun-dry-mh2-koregaon_811279,2027-09-30,0.0,0.0,0.0
962182,Zepto_pun-dry-mh2-koregaon_811279,2027-10-31,0.0,0.0,0.0
962183,Zepto_pun-dry-mh2-koregaon_811279,2027-11-30,0.0,0.0,0.0


In [902]:
ly_actuals_df['pri_actuals_vol_rum_lag_1'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(1)

ly_actuals_df['pri_actuals_vol_rum_lag_2'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(2)

ly_actuals_df['pri_actuals_vol_rum_lag_3'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(3)


ly_actuals_df['pri_actuals_vol_rum_lead_1'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(-1)

ly_actuals_df['pri_actuals_vol_rum_lead_2'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(-2)


In [903]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M',
        'pri_actuals_vol_rum_lag_1': 'LY Primary Actuals Lag 1 Vol',
        'pri_actuals_vol_rum_lag_2': 'LY Primary Actuals Lag 2 Vol',
        'pri_actuals_vol_rum_lag_3': 'LY Primary Actuals Lag 3 Vol',
        'pri_actuals_vol_rum_lead_1': 'LY Primary Actuals Lead 1 Vol',
        'pri_actuals_vol_rum_lead_2': 'LY Primary Actuals Lead 2 Vol',
    }),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [904]:
final_df[final_df['Contribution'].isna()]

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,Sec Actuals Vol,Primary P3M redundant,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [905]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [906]:
final_offtakes_historical_df = offtakes_monthly_df.copy()

In [907]:
final_offtakes_historical_df

,month_date,key,chain,parent_material_code,material_group_code,run_month,offtake_vol_rum,imputed
0,2023-01-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-01-31,22.584,NaN
1,2023-02-28,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-02-28,17.568,NaN
2,2023-03-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-03-31,25.332,NaN
3,2023-04-30,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-04-30,21.888,NaN
4,2023-05-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-05-31,15.860,NaN
...,...,...,...,...,...,...,...,...
29554,2026-08-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-08-31,0.000,NaN
29555,2026-09-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-09-30,0.000,NaN
29556,2026-10-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-10-31,0.000,NaN
29557,2026-11-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-11-30,0.000,NaN


In [908]:
final_offtakes_historical_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [909]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df[['chain', 'parent_material_code']].astype(str).agg(
    '_'.join, axis=1
)

In [910]:

final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [911]:
final_offtakes_historical_df['P3M'] = final_offtakes_historical_df.groupby(
    ['key'], as_index = False, group_keys = False)['offtake_vol_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

In [912]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M']].rename(
        columns={'P3M': 'Offtake P3M', 'offtake_vol_rum': 'Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [913]:
ly_final_offtakes_historical_df = final_offtakes_historical_df.copy()

In [914]:
ly_final_offtakes_historical_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [915]:
ly_final_offtakes_historical_df['month_date'] = ly_final_offtakes_historical_df['month_date'] + MonthEnd(12)
ly_final_offtakes_historical_df.head()

,month_date,key,chain,parent_material_code,material_group_code,run_month,offtake_vol_rum,imputed,P3M
0,2024-01-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-01-31,22.584,NaN,NaN
1,2024-02-29,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-02-28,17.568,NaN,NaN
2,2024-03-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-03-31,25.332,NaN,NaN
3,2024-04-30,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-04-30,21.888,NaN,21.828
4,2024-05-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-05-31,15.860,NaN,21.596


In [916]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lag 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 3 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(3)


In [917]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lead 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lead 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-2)

In [918]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M', 'LY Offtake Actuals Lag 1 Vol', 
                                     'LY Offtake Actuals Lag 2 Vol', 
                                     'LY Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lead 1 Vol',
                                     'LY Offtake Actuals Lead 2 Vol']].rename(
        columns={'P3M': 'LY Offtake P3M', 'offtake_vol_rum': 'LY Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [919]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [920]:
for col in ['Primary P3M', 'LY Primary P3M', 'Offtake P3M', 'LY Offtake P3M', 'Primary P3M redundant']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key2'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [921]:
final_offtakes_historical_df

,month_date,key,chain,parent_material_code,material_group_code,run_month,offtake_vol_rum,imputed,P3M
0,2023-01-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-01-31,22.584,NaN,NaN
1,2023-02-28,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-02-28,17.568,NaN,NaN
2,2023-03-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-03-31,25.332,NaN,NaN
3,2023-04-30,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-04-30,21.888,NaN,21.828
4,2023-05-31,Blinkit_718288,Blinkit,718288,SAFF GOLD,2023-05-31,15.860,NaN,21.596
...,...,...,...,...,...,...,...,...,...
29554,2026-08-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-08-31,0.000,NaN,0.000
29555,2026-09-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-09-30,0.000,NaN,0.000
29556,2026-10-31,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-10-31,0.000,NaN,0.000
29557,2026-11-30,Zepto_811287,Zepto,811287,PA_RSW_SR,2026-11-30,0.000,NaN,0.000


In [922]:
final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)

In [923]:
# final_offtakes_historical_df['OT_Lag_1'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(1)

# final_offtakes_historical_df['OT_Lag_2'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(2)

# final_offtakes_historical_df['OT_Lag_3'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(3)

In [924]:
# del final_df['OT_Lag_1'], final_df['OT_Lag_2'], final_df['OT_Lag_3']

In [925]:
# len_before_merge = len(final_df)
# final_df = final_df.merge(
#     final_offtakes_historical_df[['key', 'month_date', 'OT_Lag_1', 'OT_Lag_2', 'OT_Lag_3']].rename(
#         columns={'month_date': 'run_month'}
#     ),
#     on=['key', 'run_month'],
#     how='left'
# )
# assert len_before_merge == len(final_df)
# del len_before_merge

In [926]:
lags_df = plan_actuals_df.copy()

In [927]:
lags_df.sort_values(by=['key2', 'month_date'], inplace=True)

In [928]:
lags_df

,month_date,key2,chain,facility_name,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2026-06-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.021845,0.0,0.0,NaN
1,2026-07-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
2,2026-08-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
3,2026-09-30,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
4,2026-10-31,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
962180,2026-08-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962181,2026-09-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962182,2026-10-31,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0
962183,2026-11-30,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,0.0,0.000000,0.0,0.0,0.0


In [929]:
lags_df['Primary_Lag_2'] = lags_df.groupby(
    ['material_group_code', 'key2']
)['pri_actuals_vol_rum'].shift(1)

lags_df['Primary_Lag_3'] = lags_df.groupby(
    ['material_group_code', 'key2']
)['pri_actuals_vol_rum'].shift(2)

In [930]:
lags_df['month_date'] = lags_df['month_date'] + MonthEnd(1)

In [931]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_df[['key2', 'month_date', 'pri_actuals_vol_rum', 'Primary_Lag_2', 'Primary_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'pri_actuals_vol_rum': 'Primary_Lag_1'
    }),
    on=['key2', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [932]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Vol,LY Offtake P3M,LY Offtake Actuals Lag 1 Vol,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


In [933]:
lags_final_offtakes_historical_df = final_offtakes_historical_df.copy()

lags_final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)
lags_final_offtakes_historical_df['OT_Lag_2'] = lags_final_offtakes_historical_df.groupby(
    ['material_group_code', 'key']
)['offtake_vol_rum'].shift(1)

lags_final_offtakes_historical_df['OT_Lag_3'] = lags_final_offtakes_historical_df.groupby(
    ['material_group_code', 'key']
)['offtake_vol_rum'].shift(2)

In [934]:
# del final_df['OT_Lag_1'], final_df['OT_Lag_2'], final_df['OT_Lag_3']

In [935]:
lags_final_offtakes_historical_df['month_date'] = lags_final_offtakes_historical_df['month_date'] + MonthEnd(1)

In [936]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'offtake_vol_rum': 'OT_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [937]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [938]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [939]:
final_df.columns

Index(['key2', 'chain', 'facility_name', 'parent_material_code',
       'material_group_code', 'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'key', 'Offtake Chain PSKU',
       'Primary Actuals Chain PSKU P3M Sum Vol', 'Group Size',
       'Fallback Contribution', 'Contribution', 'Final Contribution',
       'Offtake Chain FC PSKU', 'M month', 'Offtake Chain PSKU Forecast Vol',
       'Offtake Chain FC PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY P

In [940]:
final_df['Offtake Chain FC PSKU Lag 1 Vol'] = final_df['OT_Lag_1'] * final_df['Final Contribution']
final_df['Offtake Chain FC PSKU Lag 2 Vol'] = final_df['OT_Lag_2'] * final_df['Final Contribution']
final_df['Offtake Chain FC PSKU Lag 3 Vol'] = final_df['OT_Lag_3'] * final_df['Final Contribution']

final_df['Offtake Chain FC PSKU P3M Vol'] = final_df['Offtake P3M'] * final_df['Final Contribution']
final_df['LY Offtake Chain FC PSKU P3M Vol'] = final_df['LY Offtake P3M'] * final_df['Final Contribution']

final_df['LY Offtake Chain FC PSKU Lag 1 Vol'] = final_df['LY Offtake Actuals Lag 1 Vol'] * final_df['Final Contribution']
final_df['LY Offtake Chain FC PSKU Lag 2 Vol'] = final_df['LY Offtake Actuals Lag 2 Vol'] * final_df['Final Contribution']
final_df['LY Offtake Chain FC PSKU Lag 3 Vol'] = final_df['LY Offtake Actuals Lag 3 Vol'] * final_df['Final Contribution']

final_df['LY Offtake Chain FC PSKU Lead 1 Vol'] = final_df['LY Offtake Actuals Lead 1 Vol'] * final_df['Final Contribution']
final_df['LY Offtake Chain FC PSKU Lead 2 Vol'] = final_df['LY Offtake Actuals Lead 2 Vol'] * final_df['Final Contribution']

In [941]:
final_df['LY Offtake Chain FC PSKU Actuals Vol'] = final_df['LY Offtake Actuals Vol'] * final_df['Final Contribution']

In [942]:
final_df

,key2,chain,facility_name,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,Offtake Chain FC PSKU Lag 2 Vol,Offtake Chain FC PSKU Lag 3 Vol,Offtake Chain FC PSKU P3M Vol,LY Offtake Chain FC PSKU P3M Vol,LY Offtake Chain FC PSKU Lag 1 Vol,LY Offtake Chain FC PSKU Lag 2 Vol,LY Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Chain FC PSKU Lead 1 Vol,LY Offtake Chain FC PSKU Lead 2 Vol,LY Offtake Chain FC PSKU Actuals Vol
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [943]:
final_df.shape

(404430, 67)

### Format

In [944]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [945]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,TRU_PDRFR,850.57000
1,2027-03-31,TRU_OATS,177.07000
2,2027-03-31,TRU_QUINO,204.75000
3,2027-03-31,TRU_RAW,453.44000
4,2027-03-31,NHR_NHO_E,266.57953


In [946]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [947]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [948]:
final_df.columns

Index(['key2', 'chain', 'facility_name', 'parent_material_code',
       'material_group_code', 'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'key', 'Offtake Chain PSKU',
       'Primary Actuals Chain PSKU P3M Sum Vol', 'Group Size',
       'Fallback Contribution', 'Contribution', 'Final Contribution',
       'Offtake Chain FC PSKU', 'M month', 'Offtake Chain PSKU Forecast Vol',
       'Offtake Chain FC PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY P

In [949]:
final_df = final_df.rename(columns={
    'key2': 'Key2',
    'key': 'Key',
    'chain': 'Chain',
    'facility_name': 'FC',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'run_month': 'Run Month',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Till Date Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol',
    'Primary P3M': 'Primary P3M Vol',
    'Offtake Chain PSKU': 'Offtake Chain PSKU Vol',
    'Offtake Chain FC PSKU': 'Offtake Chain FC PSKU Vol',
    'norms_soh': 'Norms SOH',
    'norm_days': 'Norm Days',
    'safety_stock': 'Safety Stock Vol',
    'Actual Closing SOH': 'Actual Closing SOH Vol',
    'Actual Closing SOH_Lag_1': 'Actual Closing SOH Lag 1 Vol',
    'Actual Closing SOH Lag 2': 'Actual Closing SOH Lag 2 Vol',
    'Assumed Closing SOH': 'Assumed Closing SOH Vol',
    'Assumed Closing SOH_Lag_1': 'Assumed Closing SOH Lag 1 Vol',
    'Assumed Closing SOH Lag 2': 'Assumed Closing SOH Lag 2 Vol',
    'Primary P3M redundant': 'Primary P3M redundant Vol',
    'LY Primary P3M': 'LY Primary P3M Vol',
    'Offtake P3M': 'Offtake P3M Vol',
    'LY Offtake P3M': 'LY Offtake P3M Vol',
    'Primary_Lag_1': 'Primary Actuals Lag 1 Vol',
    'Primary_Lag_2': 'Primary Actuals Lag 2 Vol',
    'Primary_Lag_3': 'Primary Actuals Lag 3 Vol',
    'OT_Lag_1': 'Offtake Actuals Lag 1 Vol',
    'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
    'OT_Lag_3': 'Offtake Actuals Lag 3 Vol',
    'qtr_ind_rate': 'Index Rate',
    'portfolio': 'Portfolio'
})

In [950]:
final_df.columns

Index(['Key2', 'Chain', 'FC', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain PSKU Vol',
       'Primary Actuals Chain PSKU P3M Sum Vol', 'Group Size',
       'Fallback Contribution', 'Contribution', 'Final Contribution',
       'Offtake Chain FC PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Norms SOH', 'Norm Days', 'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lea

In [951]:
columns_order = [
    'Key2', 'Key', 'Chain', 'FC', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 
    'Run Month', 'Month Date', 'M month', 
    
    'Primary Till Date Actuals Vol', 'Secondary Plan Vol', 'Primary P3M Vol', 
    'Primary Actuals Chain PSKU P3M Sum Vol', 
    
    'Group Size', 'Fallback Contribution', 'Contribution', 'Final Contribution',

    'Offtake Chain PSKU Vol', 'Offtake Chain FC PSKU Vol', 'Offtake Chain PSKU Forecast Vol', 
    'Offtake Chain FC PSKU Forecast Vol',
    
    'Norms SOH', 'Norm Days', 'Safety Stock Vol', 'Actual Closing SOH Vol',
    'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol', 'Assumed Closing SOH Vol',
    'Assumed Closing SOH Lag 1 Vol', 'Assumed Closing SOH Lag 2 Vol', 
    'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
    
    'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
    'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',

    'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol', 
    
    'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 
    'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',

    'Offtake Actuals Lag 1 Vol', 'Offtake Actuals Lag 2 Vol',
    'Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lag 1 Vol', 
    'LY Offtake Actuals Lag 2 Vol', 'LY Offtake Actuals Lag 3 Vol',
    'LY Offtake Actuals Lead 1 Vol', 'LY Offtake Actuals Lead 2 Vol',

    'Offtake Chain FC PSKU Lag 1 Vol', 'Offtake Chain FC PSKU Lag 2 Vol', 
    'Offtake Chain FC PSKU Lag 3 Vol', 'Offtake Chain FC PSKU P3M Vol', 
    'LY Offtake Chain FC PSKU P3M Vol', 'LY Offtake Chain FC PSKU Actuals Vol',
    'LY Offtake Chain FC PSKU Lag 1 Vol', 'LY Offtake Chain FC PSKU Lag 2 Vol', 
    'LY Offtake Chain FC PSKU Lag 3 Vol', 'LY Offtake Chain FC PSKU Lead 1 Vol',
    'LY Offtake Chain FC PSKU Lead 2 Vol',

    'Calculated Primary Vol'
]

In [952]:
for col in final_df.columns:
    try:
        assert col in columns_order
    except:
        print(col)

In [953]:
final_df.columns

Index(['Key2', 'Chain', 'FC', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain PSKU Vol',
       'Primary Actuals Chain PSKU P3M Sum Vol', 'Group Size',
       'Fallback Contribution', 'Contribution', 'Final Contribution',
       'Offtake Chain FC PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Norms SOH', 'Norm Days', 'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lea

In [954]:
final_df

,Key2,Chain,FC,PSKU,Brand,Run Month,Month Date,Primary Till Date Actuals Vol,Secondary Plan Vol,Primary P3M Vol,...,Offtake Chain FC PSKU P3M Vol,LY Offtake Chain FC PSKU P3M Vol,LY Offtake Chain FC PSKU Lag 1 Vol,LY Offtake Chain FC PSKU Lag 2 Vol,LY Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Chain FC PSKU Lead 1 Vol,LY Offtake Chain FC PSKU Lead 2 Vol,LY Offtake Chain FC PSKU Actuals Vol,Index Rate,Portfolio
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-05-31,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-06-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-07-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),2026-06-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-10-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260000.00000,Saffola Oils
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-11-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260000.00000,Saffola Oils
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2026-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260000.00000,Saffola Oils
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,2026-06-30,2027-01-31,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260000.00000,Saffola Oils


In [955]:
def calculate_primary_iteratively(key_df):
    key_df = key_df.sort_values('Month Date').copy()
    key_df = key_df[key_df['Month Date'] >= key_df['Run Month']]

    # Fill NaNs
    fill_cols = [
        'Offtake Chain FC PSKU Forecast Vol',
        'Safety Stock Vol',
        'Actual Closing SOH Lag 1 Vol'
    ]
    key_df[fill_cols] = key_df[fill_cols].fillna(0)

    _key = key_df['Key2'].iloc[0]
    _run_month = key_df['Run Month'].iloc[0]

    outputs = []
    prev_soh = None

    for _, row in key_df.iterrows():
        forecast = row['Offtake Chain FC PSKU Forecast Vol']
        safety_stock = row['Safety Stock Vol']

        if row['M month'] == 'M':
            opening_soh = row['Actual Closing SOH Lag 1 Vol']
        else:
            opening_soh = prev_soh

        primary_vol = max(
            forecast + safety_stock - opening_soh,
            0
        )

        assumed_closing_soh = max(
            opening_soh + primary_vol - forecast,
            0
        )

        outputs.append({
            'Run Month': _run_month,
            'Key2': _key,
            'Month Date': row['Month Date'],
            'Calculated Primary Vol': primary_vol,
            'Final Assumed Closing SOH Vol': assumed_closing_soh
        })

        prev_soh = assumed_closing_soh

    return outputs

In [956]:
final_df.groupby(['Month Date'])[['Offtake Chain FC PSKU Forecast Vol',
        'Safety Stock Vol',
        'Actual Closing SOH Lag 1 Vol']].sum().reset_index()

,Month Date,Offtake Chain FC PSKU Forecast Vol,Safety Stock Vol,Actual Closing SOH Lag 1 Vol
0,2026-05-31,0.000000,33305.101770,0.000000
1,2026-06-30,141918.580798,32542.571700,51207.912095
2,2026-07-31,144518.371383,33551.712449,0.000000
3,2026-08-31,149201.348255,33810.177610,0.000000
4,2026-09-30,145968.869091,37296.111423,0.000000
5,2026-10-31,164876.642568,48854.331553,0.000000
6,2026-11-30,206742.674655,48988.134589,0.000000
7,2026-12-31,212785.157225,46965.068543,0.000000
8,2027-01-31,202003.295625,3222.705702,0.000000
9,2027-02-28,16712.322882,0.000000,0.000000


In [957]:
# ### Move UP

# final_df['Calculated Primary Vol'] = np.where(
#     final_df['M month'] == 'M',
#     final_df['Secondary Plan Vol'],
#     final_df['Offtake Chain FC PSKU Forecast Vol'].fillna(0) + \
#         final_df['Safety Stock Vol'].fillna(0) - \
#         final_df['Assumed Closing SOH Lag 1 Vol'].fillna(0)
# )

In [958]:
calculated_primary = []

for (_, _), group_df in tqdm(final_df.groupby(['Run Month', 'Key2'])):
    calculated_primary.extend(
        calculate_primary_iteratively(group_df)
    )

calculated_primary_df = pd.DataFrame(calculated_primary)

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 40443/40443 [01:32<00:00, 439.04it/s]


In [959]:
calculated_primary_df['Final Assumed Closing SOH Vol'].sum()

521825.65863061737

In [960]:
del calculated_primary

In [961]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    calculated_primary_df,
    on=['Run Month', 'Month Date', 'Key2'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [962]:
final_df['Calculated Primary Vol'].min(), final_df['Final Assumed Closing SOH Vol'].min()

(0.0, 0.0)

In [963]:
final_df.sort_values(by=['Run Month', 'Key2', 'Month Date'], inplace=True)

In [964]:
final_df['Final Assumed Closing SOH Lag 1 Vol'] = final_df.groupby(
    ['Run Month', 'Key']
)['Final Assumed Closing SOH Vol'].shift(1)

In [965]:
final_df.columns

Index(['Key2', 'Chain', 'FC', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain PSKU Vol',
       'Primary Actuals Chain PSKU P3M Sum Vol', 'Group Size',
       'Fallback Contribution', 'Contribution', 'Final Contribution',
       'Offtake Chain FC PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Norms SOH', 'Norm Days', 'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lea

In [966]:
final_df = final_df[columns_order]

In [967]:
vol_to_val_cols = [col for col in final_df.columns if 'Vol' in col]
vol_to_val_cols

['Primary Till Date Actuals Vol',
 'Secondary Plan Vol',
 'Primary P3M Vol',
 'Primary Actuals Chain PSKU P3M Sum Vol',
 'Offtake Chain PSKU Vol',
 'Offtake Chain FC PSKU Vol',
 'Offtake Chain PSKU Forecast Vol',
 'Offtake Chain FC PSKU Forecast Vol',
 'Safety Stock Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Primary Actuals Vol',
 'Sec Actuals Vol',
 'Primary P3M redundant Vol',
 'LY Primary Actuals Vol',
 'LY Sec Actuals Vol',
 'LY Primary P3M Vol',
 'Offtake Actuals Vol',
 'Offtake P3M Vol',
 'LY Offtake Actuals Vol',
 'LY Offtake P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',


In [968]:
for col in vol_to_val_cols:
    final_df[col[:-3] + 'Val'] = final_df[col].fillna(0) * final_df['Index Rate'] / (10 ** 7)

In [969]:
final_df.duplicated(subset=['Run Month', 'Key2', 'Month Date']).sum()

0

### Depot Mappings

In [72]:
facility_to_city_mappings_df = pd.read_excel(
    r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
    'Sheet1'
)

In [73]:
facility_to_city_mappings_df.columns = facility_to_city_mappings_df.columns.str.lower()

In [74]:
facility_to_city_mappings_df.columns = ['chain', 'facility_name', 'city', 'customer', 'marico_depot',
       'status', 'depot_name']

In [75]:
facility_to_city_mappings_df = facility_to_city_mappings_df[
    facility_to_city_mappings_df['customer'].notna()
]

In [76]:
facility_to_city_mappings_df['facility_name'] = facility_to_city_mappings_df['facility_name'].str.lower()
facility_to_city_mappings_df['city'] = facility_to_city_mappings_df['city'].str.lower()

In [77]:
facility_to_city_mappings_df.head()

,chain,facility_name,city,customer,marico_depot,status,depot_name
0,Blinkit,farukhnagar f2 - feeder warehouse,gurugram,16012,D115,Active,NaN
1,Blinkit,ahmedabad a2 - feeder warehouse,ahmedabad,15616,D354,Active,NaN
2,Blinkit,hyderabad h3 - feeder warehouse,hyderabad,16005,D530,Active,NaN
3,Blinkit,lucknow l5 - feeder warehouse,lucknow,15855,D113,Active,NaN
4,Blinkit,super store hyderabad h2 - warehouse,hyderabad,9895,D530,Active,NaN


In [78]:
customer_depot_mappings_df = pd.read_sql("""
SELECT DISTINCT customer_code, depot_code, channel_name 
FROM mst_customer
WHERE company_code='MIL' AND
    latest_record_ind=1
ORDER BY 3, 1, 2
""",
prod_conn
)
customer_depot_mappings_df.columns = customer_depot_mappings_df.columns.str.lower()

In [79]:
customer_depot_mappings_df.duplicated(subset=['customer_code']).sum()

0

In [80]:
customer_depot_mappings_df['customer_code'] = customer_depot_mappings_df['customer_code'].astype(str)
facility_to_city_mappings_df['customer'] = facility_to_city_mappings_df['customer'].astype(str)

In [81]:
customer_depot_mappings_df.dtypes

customer_code    object
depot_code       object
channel_name     object
dtype: object

In [82]:
facility_to_city_mappings_df.dtypes

chain             object
facility_name     object
city              object
customer          object
marico_depot      object
status            object
depot_name       float64
dtype: object

In [83]:
len_before_merge = len(facility_to_city_mappings_df)
facility_to_city_mappings_df = facility_to_city_mappings_df.merge(
    customer_depot_mappings_df[['customer_code', 'depot_code']].drop_duplicates().rename(
        columns={'customer_code': 'customer'}
    ),
    on=['customer'],
    how='left'
)
assert len_before_merge == len(facility_to_city_mappings_df)
del len_before_merge

In [982]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    facility_to_city_mappings_df[['facility_name', 'depot_code']].drop_duplicates().rename(
        columns={'depot_code': 'Depot', 'facility_name': 'FC'}
    ),
    on=['FC'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [983]:
final_df[['Index Rate', 'Portfolio']].isna().sum()

Index Rate      190
Portfolio     26670
dtype: int64

In [984]:
final_df.columns

Index(['Key2', 'Key', 'Chain', 'FC', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain FC PSKU P3M Val', 'LY Offtake Chain FC PSKU P3M Val',
       'LY Offtake Chain FC PSKU Actuals Val',
       'LY Offtake Chain FC PSKU Lag 1 Val',
       'LY Offtake Chain FC PSKU Lag 2 Val',
       'LY Offtake Chain FC PSKU Lag 3 Val',
       'LY Offtake Chain FC PSKU Lead 1 Val',
       'LY Offtake Chain FC PSKU Lead 2 Val', 'Calculated Primary Val',
       'Depot'],
      dtype='object', length=128)

In [985]:
final_df['Offtake Chain FC PSKU Forecast Vol'].sum()

1384727.2624818403

In [986]:
final_df[final_df['Depot'].isna()].to_csv('chek_nan.csv')#['Offtake Chain FC PSKU Forecast Vol'].sum()

### Depot PSKU

In [987]:
offtake_columns = [
    col for col in final_df.columns 
    if ('offtake' in col.lower()) &
    ('vol' in col.lower()) & ('fc' in col.lower())
]

for col in ['Offtake Chain PSKU Vol', 'Offtake Chain FC PSKU Vol',  'Offtake Actuals Vol']:
    if col in offtake_columns:
        offtake_columns.remove(col)
    
offtake_columns

['Offtake Chain FC PSKU Forecast Vol',
 'Offtake Chain FC PSKU Lag 1 Vol',
 'Offtake Chain FC PSKU Lag 2 Vol',
 'Offtake Chain FC PSKU Lag 3 Vol',
 'Offtake Chain FC PSKU P3M Vol',
 'LY Offtake Chain FC PSKU P3M Vol',
 'LY Offtake Chain FC PSKU Actuals Vol',
 'LY Offtake Chain FC PSKU Lag 1 Vol',
 'LY Offtake Chain FC PSKU Lag 2 Vol',
 'LY Offtake Chain FC PSKU Lag 3 Vol',
 'LY Offtake Chain FC PSKU Lead 1 Vol',
 'LY Offtake Chain FC PSKU Lead 2 Vol']

In [988]:
soh_cols = [
    col for col in final_df.columns 
    if ('soh' in col.lower()) &
    ('closing' in col.lower()) &
    ('vol' in col.lower())
]
soh_cols

['Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol']

In [989]:
final_df.columns

Index(['Key2', 'Key', 'Chain', 'FC', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain FC PSKU P3M Val', 'LY Offtake Chain FC PSKU P3M Val',
       'LY Offtake Chain FC PSKU Actuals Val',
       'LY Offtake Chain FC PSKU Lag 1 Val',
       'LY Offtake Chain FC PSKU Lag 2 Val',
       'LY Offtake Chain FC PSKU Lag 3 Val',
       'LY Offtake Chain FC PSKU Lead 1 Val',
       'LY Offtake Chain FC PSKU Lead 2 Val', 'Calculated Primary Val',
       'Depot'],
      dtype='object', length=128)

In [990]:
base_df = final_df.copy()
base_df = base_df[
    base_df['Depot'].notna()
]

In [991]:
base_df.shape

(394850, 128)

In [992]:
base_df['FC'].isna().sum()

0

In [993]:
base_df = base_df.groupby(
    ['Run Month', 'Month Date', 'M month', 'Depot', 'PSKU', 
     'Brand', 'Portfolio', 'Index Rate'], as_index=False
)[['Calculated Primary Vol'] + offtake_columns + soh_cols + ['Safety Stock Vol']].sum()

In [994]:
base_df['Offtake Chain FC PSKU Forecast Vol'].sum()

1377567.7094953395

In [995]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
WHERE month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)

In [996]:
depot_psku_primary_df.head()

,DEPOT_CODE,PARENT_MATERIAL_CODE,MATERIAL_GROUP_CODE,MONTH_DATE,PRI_ACTUALS_VOL_RUM,PRI_APO_PLAN_VOL_RUM,SEC_APO_PLAN_VOL_RUM,SEC_ACTUALS_VOL_RUM
0,D112,718472,ADV-AHO-R,2023-02-28,0.0,0.0,0.044,0.0
1,D112,718472,ADV-AHO-R,2023-03-31,0.0,0.0,0.027,0.0
2,D112,718472,ADV-AHO-R,2023-04-30,0.0,0.0,0.044,0.0
3,D112,718472,ADV-AHO-R,2023-06-30,0.0,0.0,0.147,0.0
4,D112,718472,ADV-AHO-R,2023-07-31,0.0,0.0,0.056,0.0


In [997]:
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [998]:
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')

In [999]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [1000]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]

In [1001]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']

In [1002]:
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()

0

In [1003]:
depot_psku_primary_df = impute_missing_dates(
    depot_psku_primary_df.copy(),
    key=['depot_code', 'parent_material_code'],
    date_col='month_date'
)

11196it [00:02, 5064.92it/s]


In [1004]:
cols = ['depot_code', 'parent_material_code', 'material_group_code']

depot_psku_primary_df[cols] = depot_psku_primary_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [1005]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].fillna(0)

In [1006]:
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [1007]:
depot_psku_primary_df.duplicated(subset=['key', 'month_date']).sum()

0

In [1008]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [1009]:
depot_psku_primary_df.sort_values(by=['key', 'month_date'], inplace=True)

In [1010]:
depot_psku_primary_df['Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [1011]:
depot_psku_primary_df['Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

depot_psku_primary_df['Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

depot_psku_primary_df['Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)

depot_psku_primary_df['LY Primary Actuals Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(12)

depot_psku_primary_df['LY Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(13)

depot_psku_primary_df['LY Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(14)

depot_psku_primary_df['LY Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(15)

depot_psku_primary_df['LY Primary Actuals Lead 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(11)

depot_psku_primary_df['LY Primary Actuals Lead 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(10)

In [1012]:
depot_psku_primary_df['LY Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['Primary P3M Vol'].shift(12)

In [1013]:
depot_psku_primary_df = depot_psku_primary_df.rename(columns={
    'key': 'Key',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Actuals Vol',
    'pri_apo_plan_vol_rum': 'Primary Plan Vol',
    'sec_actuals_vol_rum': 'Secondary Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol'
})

In [1014]:
final_depot_psku_df = depot_psku_primary_df[['Key', 'Depot', 'PSKU', 'Brand']].drop_duplicates()

In [1015]:
tmp_df = pd.DataFrame()

for rm in ['2026-06-30']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_depot_psku_df.copy()
        tmp_df2['Run Month'] = pd.to_datetime(rm)
        tmp_df2['Month Date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_depot_psku_df = tmp_df.copy()    
del tmp_df

In [1016]:
final_depot_psku_df['Month Date'].max()

Timestamp('2027-02-28 00:00:00')

In [1017]:
base_df

,Run Month,Month Date,M month,Depot,PSKU,Brand,Portfolio,Index Rate,Calculated Primary Vol,Offtake Chain FC PSKU Forecast Vol,...,LY Offtake Chain FC PSKU Lead 2 Vol,Actual Closing SOH Vol,Actual Closing SOH Lag 1 Vol,Actual Closing SOH Lag 2 Vol,Assumed Closing SOH Vol,Assumed Closing SOH Lag 1 Vol,Assumed Closing SOH Lag 2 Vol,Final Assumed Closing SOH Vol,Final Assumed Closing SOH Lag 1 Vol,Safety Stock Vol
0,2026-06-30,2026-06-30,M,D112,718287,PCNO(R),CNO,349274.001420,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
1,2026-06-30,2026-06-30,M,D112,718288,SAFF GOLD,Saffola Oils,138865.260689,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
2,2026-06-30,2026-06-30,M,D112,718297,PCNO(R),CNO,349274.001420,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
3,2026-06-30,2026-06-30,M,D112,718299,PCNO(R),CNO,349274.001420,0.088437,0.076154,...,0.03908,0.0,0.0,0.0,0.012283,0.012692,0.0000,0.012283,0.000000,0.012283
4,2026-06-30,2026-06-30,M,D112,718308,PCNO(R),CNO,349274.001420,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81328,2026-06-30,2027-02-28,M+8,D677,811169,SW_SGPRF,Male Grooming,1712.605337,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
81329,2026-06-30,2027-02-28,M+8,D677,811181,SAF_CDPRS,Saffola Oils,260000.000000,0.004583,0.005579,...,0.00000,0.0,0.0,0.0,0.000000,0.000996,0.0009,0.000000,0.000996,0.000000
81330,2026-06-30,2027-02-28,M+8,D677,811267,PA_ESS_HO,Hair Oils,12860.631072,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000
81331,2026-06-30,2027-02-28,M+8,D677,811269,PA_ESS_HO,Hair Oils,12860.631072,0.000000,0.000000,...,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000


In [1018]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    base_df.drop(['M month', 'Brand', 'Portfolio', 'Index Rate'], axis=1), 
    on=['Depot', 'PSKU', 'Run Month', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1019]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Offtake Chain FC PSKU Forecast Vol,Offtake Chain FC PSKU Lag 1 Vol,Offtake Chain FC PSKU Lag 2 Vol,...,LY Offtake Chain FC PSKU Lead 2 Vol,Actual Closing SOH Vol,Actual Closing SOH Lag 1 Vol,Actual Closing SOH Lag 2 Vol,Assumed Closing SOH Vol,Assumed Closing SOH Lag 1 Vol,Assumed Closing SOH Lag 2 Vol,Final Assumed Closing SOH Vol,Final Assumed Closing SOH Lag 1 Vol,Safety Stock Vol
0,D112_715098,D112,715098,CO_SO_PCP,2026-06-30,2026-05-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D112_715099,D112,715099,CO_SO_PCP,2026-06-30,2026-05-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D112_715100,D112,715100,CO_SO_PCP,2026-06-30,2026-05-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D112_715106,D112,715106,CO_SO_PCP,2026-06-30,2026-05-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D112_715107,D112,715107,CO_SO_PCP,2026-06-30,2026-05-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111955,D677_811169,D677,811169,SW_SGPRF,2026-06-30,2027-02-28,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0000,0.0,0.000000,0.0
111956,D677_811181,D677,811181,SAF_CDPRS,2026-06-30,2027-02-28,0.004583,0.005579,0.013711,0.002925,...,0.0,0.0,0.0,0.0,0.0,0.000996,0.0009,0.0,0.000996,0.0
111957,D677_811267,D677,811267,PA_ESS_HO,2026-06-30,2027-02-28,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0000,0.0,0.000000,0.0
111958,D677_811269,D677,811269,PA_ESS_HO,2026-06-30,2027-02-28,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0000,0.0,0.000000,0.0


In [1020]:
final_depot_psku_df['Calculated Primary Vol'] = final_depot_psku_df['Calculated Primary Vol'].fillna(0)

In [1021]:
depot_psku_primary_df.columns

Index(['Month Date', 'Key', 'Depot', 'PSKU', 'Brand', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol'],
      dtype='object')

In [1022]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    depot_psku_primary_df[['Depot', 'PSKU', 'Month Date', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 
       'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol']], 
    on=['Depot', 'PSKU', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1023]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Offtake Chain FC PSKU Lag 1 Vol', 'Offtake Chain FC PSKU Lag 2 Vol',
       'Offtake Chain FC PSKU Lag 3 Vol', 'Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU Actuals Vol',
       'LY Offtake Chain FC PSKU Lag 1 Vol',
       'LY Offtake Chain FC PSKU Lag 2 Vol',
       'LY Offtake Chain FC PSKU Lag 3 Vol',
       'LY Offtake Chain FC PSKU Lead 1 Vol',
       'LY Offtake Chain FC PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary Plan Vol', 'Secondary Plan Vol',
       'Secondary 

In [1024]:
final_depot_psku_df.sort_values(by=['Run Month', 'Key', 'Month Date'], inplace=True)

In [1025]:
for col in ['Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 
            'Primary Actuals Lag 3 Vol', 'LY Primary P3M Vol']:
    # if not 'LY' in col:  'LY P6M',
    final_depot_psku_df.loc[final_depot_psku_df['Month Date'] > final_depot_psku_df['Run Month'], [col]] = np.nan
    final_depot_psku_df[col] = final_depot_psku_df.groupby(['Run Month', 'Key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [1026]:
vol_to_val_cols = [col for col in final_depot_psku_df.columns if ('Vol' in col)]
vol_to_val_cols

['Calculated Primary Vol',
 'Offtake Chain FC PSKU Forecast Vol',
 'Offtake Chain FC PSKU Lag 1 Vol',
 'Offtake Chain FC PSKU Lag 2 Vol',
 'Offtake Chain FC PSKU Lag 3 Vol',
 'Offtake Chain FC PSKU P3M Vol',
 'LY Offtake Chain FC PSKU P3M Vol',
 'LY Offtake Chain FC PSKU Actuals Vol',
 'LY Offtake Chain FC PSKU Lag 1 Vol',
 'LY Offtake Chain FC PSKU Lag 2 Vol',
 'LY Offtake Chain FC PSKU Lag 3 Vol',
 'LY Offtake Chain FC PSKU Lead 1 Vol',
 'LY Offtake Chain FC PSKU Lead 2 Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Safety Stock Vol',
 'Primary Actuals Vol',
 'Primary Plan Vol',
 'Secondary Plan Vol',
 'Secondary Actuals Vol',
 'Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Vol',
 '

In [1027]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1028]:
for col in vol_to_val_cols:
    final_depot_psku_df[col[:-3] + 'Val'] = final_depot_psku_df[col] * final_depot_psku_df['Index Rate'] / (10 ** 7)

In [1029]:
final_depot_psku_df['M Month'] = final_depot_psku_df.apply(
    lambda x: mappings[x['Run Month']].get(x['Month Date'], np.nan),
    axis=1
)

In [1030]:
final_depot_psku_df = final_depot_psku_df[final_depot_psku_df['M Month'].notna()]

In [1031]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Offtake Chain FC PSKU Lag 1 Vol', 'Offtake Chain FC PSKU Lag 2 Vol',
       'Offtake Chain FC PSKU Lag 3 Vol', 'Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU Actuals Vol',
       'LY Offtake Chain FC PSKU Lag 1 Vol',
       'LY Offtake Chain FC PSKU Lag 2 Vol',
       'LY Offtake Chain FC PSKU Lag 3 Vol',
       'LY Offtake Chain FC PSKU Lead 1 Vol',
       'LY Offtake Chain FC PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary Plan Vol', 'Secondary Plan Vol',
       'Secondary 

In [1032]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'Brand'}
    ),
    on=['Brand'], 
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1033]:
final_depot_psku_df.rename(columns={'portfolio': 'Portfolio'}, inplace=True)

In [1034]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain FC PSKU Forecast Vol',
       'Offtake Chain FC PSKU Lag 1 Vol', 'Offtake Chain FC PSKU Lag 2 Vol',
       'Offtake Chain FC PSKU Lag 3 Vol', 'Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU P3M Vol',
       'LY Offtake Chain FC PSKU Actuals Vol',
       'LY Offtake Chain FC PSKU Lag 1 Vol',
       'LY Offtake Chain FC PSKU Lag 2 Vol',
       'LY Offtake Chain FC PSKU Lag 3 Vol',
       'LY Offtake Chain FC PSKU Lead 1 Vol',
       'LY Offtake Chain FC PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary Plan Vol', 'Secondary Plan Vol',
       'Secondary 

In [1035]:
columns_order = [
    'Key', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'Index Rate', 
    'Run Month', 'Month Date', 'M Month',

    'Calculated Primary Vol', 
    
    'Secondary Plan Vol', 'Primary Actuals Lag 1 Vol',
    'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Vol', 'Primary P3M Vol', 'LY Primary P3M Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
    'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
    
    
    'Offtake Chain FC PSKU Forecast Vol',
    'Offtake Chain FC PSKU Lag 1 Vol', 'Offtake Chain FC PSKU Lag 2 Vol',
    'Offtake Chain FC PSKU Lag 3 Vol', 'LY Offtake Chain FC PSKU Actuals Vol',
    'Offtake Chain FC PSKU P3M Vol', 'LY Offtake Chain FC PSKU P3M Vol',
    'LY Offtake Chain FC PSKU Lag 1 Vol', 'LY Offtake Chain FC PSKU Lag 2 Vol',
    'LY Offtake Chain FC PSKU Lag 3 Vol', 'LY Offtake Chain FC PSKU Lead 1 Vol',
    'LY Offtake Chain FC PSKU Lead 2 Vol', 

    'Actual Closing SOH Vol', 'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol', 
    'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
    'Safety Stock Vol',
    
    'Calculated Primary Val', 

    'Secondary Plan Val', 'Primary Actuals Lag 1 Val',
    'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
    'LY Primary Actuals Val', 'Primary P3M Val', 'LY Primary P3M Val',
    'LY Primary Actuals Lag 1 Val', 'LY Primary Actuals Lag 2 Val',
    'LY Primary Actuals Lag 3 Val', 'LY Primary Actuals Lead 1 Val', 'LY Primary Actuals Lead 2 Val',

    
    'Offtake Chain FC PSKU Forecast Val',
    'Offtake Chain FC PSKU Lag 1 Val', 'Offtake Chain FC PSKU Lag 2 Val',
    'Offtake Chain FC PSKU Lag 3 Val', 'LY Offtake Chain FC PSKU Actuals Val', 
    'Offtake Chain FC PSKU P3M Val', 'LY Offtake Chain FC PSKU P3M Val', 
    'LY Offtake Chain FC PSKU Lag 1 Val', 'LY Offtake Chain FC PSKU Lag 2 Val',
    'LY Offtake Chain FC PSKU Lag 3 Val', 'LY Offtake Chain FC PSKU Lead 1 Val',
    'LY Offtake Chain FC PSKU Lead 2 Val', 
    
    'Actual Closing SOH Val', 'Actual Closing SOH Lag 1 Val', 'Actual Closing SOH Lag 2 Val', 
    'Final Assumed Closing SOH Val', 'Final Assumed Closing SOH Lag 1 Val', 'Safety Stock Val', 
]
    # 'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol', 'Assumed Closing SOH Lag 2 Vol', 
    # 'Assumed Closing SOH Val', 'Assumed Closing SOH Lag 1 Val', 'Assumed Closing SOH Lag 2 Val', 

In [1036]:
for col in final_depot_psku_df.columns:
    if col not in columns_order:
        print(col)

Assumed Closing SOH Vol
Assumed Closing SOH Lag 1 Vol
Assumed Closing SOH Lag 2 Vol
Primary Actuals Vol
Primary Plan Vol
Secondary Actuals Vol
Assumed Closing SOH Val
Assumed Closing SOH Lag 1 Val
Assumed Closing SOH Lag 2 Val
Primary Actuals Val
Primary Plan Val
Secondary Actuals Val


In [1037]:
final_depot_psku_df = final_depot_psku_df[columns_order]

In [1038]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,Calculated Primary Vol,...,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val
0,D112_715098,D112,715098,CO_SO_PCP,Skin Care,1220.081,2026-06-30,2026-06-30,M,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D112_715098,D112,715098,CO_SO_PCP,Skin Care,1220.081,2026-06-30,2026-07-31,M+1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D112_715098,D112,715098,CO_SO_PCP,Skin Care,1220.081,2026-06-30,2026-08-31,M+2,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D112_715098,D112,715098,CO_SO_PCP,Skin Care,1220.081,2026-06-30,2026-09-30,M+3,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D112_715098,D112,715098,CO_SO_PCP,Skin Care,1220.081,2026-06-30,2026-10-31,M+4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100759,D677_811279,D677,811279,SAF_CDPRS,Saffola Oils,260000.000,2026-06-30,2026-10-31,M+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100760,D677_811279,D677,811279,SAF_CDPRS,Saffola Oils,260000.000,2026-06-30,2026-11-30,M+5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100761,D677_811279,D677,811279,SAF_CDPRS,Saffola Oils,260000.000,2026-06-30,2026-12-31,M+6,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100762,D677_811279,D677,811279,SAF_CDPRS,Saffola Oils,260000.000,2026-06-30,2027-01-31,M+7,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1039]:
final_depot_psku_df.isna().sum()

Key                                        0
Depot                                      0
PSKU                                       0
Brand                                      0
Portfolio                               5508
                                       ...  
Actual Closing SOH Lag 1 Val           19431
Actual Closing SOH Lag 2 Val           19431
Final Assumed Closing SOH Val          19431
Final Assumed Closing SOH Lag 1 Val    19431
Safety Stock Val                       19431
Length: 71, dtype: int64

In [1040]:
final_df['Calculated Primary Val'].sum(),final_depot_psku_df['Calculated Primary Val'].sum()

(281.8914838923208, 281.24499477934137)

In [1041]:
final_df[~final_df['FC'].isna()]['Calculated Primary Val'].sum(
)

281.24499477934137

In [1042]:
final_df[final_df['Month Date'] == '2026-07-31']['Calculated Primary Val'].sum()

33.59134768848467

In [1043]:
final_df

,Key2,Key,Chain,FC,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain FC PSKU P3M Val,LY Offtake Chain FC PSKU P3M Val,LY Offtake Chain FC PSKU Actuals Val,LY Offtake Chain FC PSKU Lag 1 Val,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Calculated Primary Val,Depot
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461


In [1044]:
final_df['Offtake Chain FC PSKU Forecast Val'].sum(),final_depot_psku_df['Offtake Chain FC PSKU Forecast Val'].sum()

(287.98926764262836, 287.3427785296489)

### SAVE

In [1045]:
today_dt = datetime.today()
today_dt.strftime('%d-%b-%Y')

'04-Jun-2026'

In [1046]:
VERSION = 'june run'

In [1047]:
os.makedirs(f'QCOM Chain PSKU OTP Output/{VERSION}')

In [1048]:
final_depot_psku_df.to_csv(f'QCOM Chain PSKU OTP Output/{VERSION}/QCOM Depot PSKU Primary_{VERSION}_{today_dt.strftime("%d_%b_%Y")}.csv', index=False)
final_df.to_csv(f'QCOM Chain PSKU OTP Output/{VERSION}/QCOM Chain FC PSKU Primary_{VERSION}_{today_dt.strftime("%d_%b_%Y")}.csv', index=False)

### chain forecast

In [5]:
chain_forecast = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jun 2026_to_Sep 2026.csv')
chain_forecast.rename(columns = {'item_id':'item_code'}, inplace = True)
chain_forecast

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,Jun - forecast,Jul - forecast,Aug - forecast,Sep - forecast
0,3164,Mumbai M11 - Feeder Warehouse,10221228,Just Herbs 12 Chemical Free Nail Paint (Hot Re...,1385,Marico Ltd.,5271.0,Marico Ltd,24,25,38,37
1,3164,Mumbai M11 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 gm - R...,1385,Marico Ltd.,5271.0,Marico Ltd,1,1,1,1
2,3164,Mumbai M11 - Feeder Warehouse,10000367,Saffola Masala Veggie Twist Oats(Pouch)38 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,4001,5292,5392,5409
3,3164,Mumbai M11 - Feeder Warehouse,10162271,Saffola Gold Unflavored Multigrain Oats (with ...,1385,Marico Ltd.,5271.0,Marico Ltd,15,15,15,15
4,3164,Mumbai M11 - Feeder Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,665,690,785,862
...,...,...,...,...,...,...,...,...,...,...,...,...
5836,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,10,10,10
5837,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,66,64,70,68
5838,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,1,1,1
5839,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,12,10,11


In [6]:

# Define the date columns and their corresponding dates
date_mapping = {
    'Jun - forecast': '2026-06-30',
    'Jul - forecast': '2026-07-31',
    'Aug - forecast': '2026-08-31',
    'Sep - forecast': '2026-09-30'
}

# Get all non-forecast columns
id_cols = [col for col in chain_forecast.columns if col not in date_mapping.keys()]

# Unpivot the dataframe
blinkit_unpivoted = chain_forecast.melt(
    id_vars=id_cols,
    value_vars=list(date_mapping.keys()),
    var_name='forecast_month',
    value_name='forecast_quantity'
)

# Map the forecast month to actual dates
blinkit_unpivoted['date'] = blinkit_unpivoted['forecast_month'].map(date_mapping)
blinkit_unpivoted['date'] = pd.to_datetime(blinkit_unpivoted['date'])

# Drop the temporary forecast_month column
blinkit_unpivoted = blinkit_unpivoted.drop('forecast_month', axis=1)

blinkit_unpivoted

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,3164,Mumbai M11 - Feeder Warehouse,10221228,Just Herbs 12 Chemical Free Nail Paint (Hot Re...,1385,Marico Ltd.,5271.0,Marico Ltd,24,2026-06-30
1,3164,Mumbai M11 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 gm - R...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-06-30
2,3164,Mumbai M11 - Feeder Warehouse,10000367,Saffola Masala Veggie Twist Oats(Pouch)38 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,4001,2026-06-30
3,3164,Mumbai M11 - Feeder Warehouse,10162271,Saffola Gold Unflavored Multigrain Oats (with ...,1385,Marico Ltd.,5271.0,Marico Ltd,15,2026-06-30
4,3164,Mumbai M11 - Feeder Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,665,2026-06-30
...,...,...,...,...,...,...,...,...,...,...
23359,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-09-30
23360,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,68,2026-09-30
23361,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30
23362,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,2026-09-30


In [7]:
blinkit_unpivoted['chain_name'] = 'Blinkit'

In [8]:
chain_forecast_swiggy = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED.xlsx')
chain_forecast_swiggy.columns = chain_forecast_swiggy.columns.str.lower()
chain_forecast_swiggy

,item_code,sku_name,company,brand,city,wh_name,jun_buy_qty,jul_buy_qty,aug_buy_qty
0,10104,Parachute Advansed Ayurvedic Hot Oil warming C...,MARICO LIMITED,Parachute Advansed,HYDERABAD,HYD IM2,0,0,36
1,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,0,40,56
2,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,VIZAG,VIZ IM1,0,48,44
3,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,BANGALORE,BLR IM4,120,270,240
4,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,MUMBAI,MUM FC22,0,0,0
...,...,...,...,...,...,...,...,...,...
8364,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,CHENNAI,CHN ECOM,20,20,20
8365,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,VIZAG,VIZ IM1,20,20,40
8366,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,HYDERABAD,HYD IM4,24,24,24
8367,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM4,1,2,2


In [9]:

# Define the date columns and their corresponding dates
date_mapping = {
    'jun_buy_qty': '2026-06-30',
    'jul_buy_qty': '2026-07-31',
    'aug_buy_qty': '2026-08-31'
}

# Get all non-forecast columns
id_cols = [col for col in chain_forecast_swiggy.columns if col not in date_mapping.keys()]

# Unpivot the dataframe
swiggy_unpivoted = chain_forecast_swiggy.melt(
    id_vars=id_cols,
    value_vars=list(date_mapping.keys()),
    var_name='forecast_month',
    value_name='forecast_quantity'
)

# Map the forecast month to actual dates
swiggy_unpivoted['date'] = swiggy_unpivoted['forecast_month'].map(date_mapping)
swiggy_unpivoted['date'] = pd.to_datetime(swiggy_unpivoted['date'])

# Drop the temporary forecast_month column
swiggy_unpivoted = swiggy_unpivoted.drop('forecast_month', axis=1)

swiggy_unpivoted

,item_code,sku_name,company,brand,city,wh_name,forecast_quantity,date
0,10104,Parachute Advansed Ayurvedic Hot Oil warming C...,MARICO LIMITED,Parachute Advansed,HYDERABAD,HYD IM2,0,2026-06-30
1,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,0,2026-06-30
2,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,VIZAG,VIZ IM1,0,2026-06-30
3,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,BANGALORE,BLR IM4,120,2026-06-30
4,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,MUMBAI,MUM FC22,0,2026-06-30
...,...,...,...,...,...,...,...,...
25102,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,CHENNAI,CHN ECOM,20,2026-08-31
25103,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,VIZAG,VIZ IM1,40,2026-08-31
25104,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,HYDERABAD,HYD IM4,24,2026-08-31
25105,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM4,2,2026-08-31


In [10]:
swiggy_unpivoted['chain_name'] = 'Swiggy'
swiggy_unpivoted.rename(columns = {'wh_name':'facility_name'}, inplace = True)
swiggy_unpivoted

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date,chain_name
0,10104,Parachute Advansed Ayurvedic Hot Oil warming C...,MARICO LIMITED,Parachute Advansed,HYDERABAD,HYD IM2,0,2026-06-30,Swiggy
1,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,0,2026-06-30,Swiggy
2,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,VIZAG,VIZ IM1,0,2026-06-30,Swiggy
3,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,BANGALORE,BLR IM4,120,2026-06-30,Swiggy
4,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,MUMBAI,MUM FC22,0,2026-06-30,Swiggy
...,...,...,...,...,...,...,...,...,...
25102,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,CHENNAI,CHN ECOM,20,2026-08-31,Swiggy
25103,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,VIZAG,VIZ IM1,40,2026-08-31,Swiggy
25104,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,HYDERABAD,HYD IM4,24,2026-08-31,Swiggy
25105,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM4,2,2026-08-31,Swiggy


In [11]:
blinkit_unpivoted.columns

Index(['facility_id', 'facility_name', 'item_code', 'item_name',
       'manufacturer_id', 'manufacturer_name', 'vendor_id', 'vendor_name',
       'forecast_quantity', 'date', 'chain_name'],
      dtype='object')

In [12]:
blinkit_unpivoted = blinkit_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]
swiggy_unpivoted = swiggy_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]


In [14]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Mumbai M11 - Feeder Warehouse,10221228,2026-06-30,24
1,Blinkit,Mumbai M11 - Feeder Warehouse,10000063,2026-06-30,1
2,Blinkit,Mumbai M11 - Feeder Warehouse,10000367,2026-06-30,4001
3,Blinkit,Mumbai M11 - Feeder Warehouse,10162271,2026-06-30,15
4,Blinkit,Mumbai M11 - Feeder Warehouse,10010052,2026-06-30,665
...,...,...,...,...,...
25102,Swiggy,CHN ECOM,987028,2026-08-31,20
25103,Swiggy,VIZ IM1,987028,2026-08-31,40
25104,Swiggy,HYD IM4,991861,2026-08-31,24
25105,Swiggy,BLR IM4,999977,2026-08-31,2


In [15]:
mapping = pd.read_excel("/data/aman_singh/mt_forecast/Daily Offtake Tracker - Apr'26.xlsb",sheet_name = 'Mapping')


In [16]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [ ]:
# # Keys to match rows on
# keys = ["platform_name", "asin", "EAN", "PSKU", "UOM", "Vol per unit"]

# # Build a small DataFrame with the rows to drop
# rows_to_drop = pd.DataFrame([
#     {
#         "platform_name": "Zepto",
#         "asin": "0523a4ba-32cf-4e59-abd8-0e4086859b39",
#         "EAN": "8901088205924",
#         "PSKU": "718729",
#         "UOM": "L",
#         "Vol per unit": 100.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "197827dc-3184-4c57-a966-5461967bcb3a",
#         "EAN": "8901088884402",
#         "PSKU": "808485",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8906051370753",
#         "PSKU": "807069",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8901088075817",
#         "PSKU": "808262",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Swiggy",
#         "asin": "944906",
#         "EAN": "8901088150095",
#         "PSKU": "718976",
#         "UOM": "L",
#         "Vol per unit": 300.0,
#     },
# ])

# # Mark rows to drop via left-merge on keys
# _marked = temp.merge(
#     rows_to_drop.assign(_drop=1),
#     on=keys,
#     how="left",
#     validate="m:m"  # remove if unsure about duplicates
# )

# # Keep everything that was not marked to drop
# temp_clean = _marked[_marked["_drop"].isna()].drop(columns=["_drop"])
# temp_clean
# duplicates = temp_clean[temp_clean.duplicated(subset="asin", keep=False)]
# duplicates

In [17]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [18]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Mumbai M11 - Feeder Warehouse,10221228,2026-06-30,24,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,Mumbai M11 - Feeder Warehouse,10000063,2026-06-30,1,Blinkit,10000063,8901088068789,718494,TO,38.0
2,Blinkit,Mumbai M11 - Feeder Warehouse,10000367,2026-06-30,4001,Blinkit,10000367,8901088068758,718560,TO,38.0
3,Blinkit,Mumbai M11 - Feeder Warehouse,10162271,2026-06-30,15,Blinkit,10162271,8901088735919,724428,TO,1500.0
4,Blinkit,Mumbai M11 - Feeder Warehouse,10010052,2026-06-30,665,Blinkit,10010052,8901088206273,718498,TO,400.0
...,...,...,...,...,...,...,...,...,...,...,...
48466,Swiggy,CHN ECOM,987028,2026-08-31,20,Swiggy,987028,8901088732864,732011,L,50.0
48467,Swiggy,VIZ IM1,987028,2026-08-31,40,Swiggy,987028,8901088732864,732011,L,50.0
48468,Swiggy,HYD IM4,991861,2026-08-31,24,Swiggy,991861,8906027074531,729893,L,240.0
48469,Swiggy,BLR IM4,999977,2026-08-31,2,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        13158
asin                 13158
EAN                  13158
PSKU                 13158
UOM                  13158
Vol per unit         13158
dtype: int64

In [20]:
duplicates

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Mumbai M11 - Feeder Warehouse,10221228,2026-06-30,24,NaN,NaN,NaN,NaN,NaN,NaN
5,Blinkit,Mumbai M11 - Feeder Warehouse,10123249,2026-06-30,21,NaN,NaN,NaN,NaN,NaN,NaN
19,Blinkit,Mumbai M11 - Feeder Warehouse,10221226,2026-06-30,0,NaN,NaN,NaN,NaN,NaN,NaN
20,Blinkit,Mumbai M11 - Feeder Warehouse,10231058,2026-06-30,8,NaN,NaN,NaN,NaN,NaN,NaN
30,Blinkit,Mumbai M11 - Feeder Warehouse,10241144,2026-06-30,69,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
48463,Swiggy,NAG IM1,977696,2026-08-31,24,NaN,NaN,NaN,NaN,NaN,NaN
48464,Swiggy,HYD IM1,978064,2026-08-31,0,NaN,NaN,NaN,NaN,NaN,NaN
48465,Swiggy,KOLKATA ECOM,980876,2026-08-31,0,NaN,NaN,NaN,NaN,NaN,NaN
48469,Swiggy,BLR IM4,999977,2026-08-31,2,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

474

In [26]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
26619,Swiggy,AHM DELHIVERY,554793,2026-06-30,0,Swiggy,554793,8901088886970,809042,KG,225.0
26885,Swiggy,AHM DELHIVERY,60103,2026-06-30,0,Swiggy,60103,8901088886970,809042,KG,225.0
34988,Swiggy,AHM DELHIVERY,554793,2026-07-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
35254,Swiggy,AHM DELHIVERY,60103,2026-07-31,120,Swiggy,60103,8901088886970,809042,KG,225.0
43357,Swiggy,AHM DELHIVERY,554793,2026-08-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
43623,Swiggy,AHM DELHIVERY,60103,2026-08-31,120,Swiggy,60103,8901088886970,809042,KG,225.0
23559,Swiggy,BLR DHL,298412,2026-06-30,0,Swiggy,298412,8901088171755,719162,TO,250.0
24772,Swiggy,BLR DHL,819548,2026-06-30,192,Swiggy,819548,8901088171755,719162,TO,250.0
31928,Swiggy,BLR DHL,298412,2026-07-31,0,Swiggy,298412,8901088171755,719162,TO,250.0
33141,Swiggy,BLR DHL,819548,2026-07-31,192,Swiggy,819548,8901088171755,719162,TO,250.0


In [27]:
duplicates.to_csv('duplicates_swiggy.csv')

In [30]:
df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
       .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

0

In [31]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
20095,Blinkit,Dasna D3 - Feeder Warehouse,10004412,2026-09-30,28496,Blinkit,10004412,8901088068734,718559,TO,38.0
18939,Blinkit,Farukhnagar F2 - Feeder Warehouse,10004412,2026-09-30,27782,Blinkit,10004412,8901088068734,718559,TO,38.0
14254,Blinkit,Dasna D3 - Feeder Warehouse,10004412,2026-08-31,26870,Blinkit,10004412,8901088068734,718559,TO,38.0
13098,Blinkit,Farukhnagar F2 - Feeder Warehouse,10004412,2026-08-31,25293,Blinkit,10004412,8901088068734,718559,TO,38.0
17529,Blinkit,Mumbai M11 - Feeder Warehouse,10000364,2026-09-30,23667,Blinkit,10000364,8901088034593,718398,KL,932.0
...,...,...,...,...,...,...,...,...,...,...,...
30139,Swiggy,HYD IM2,565978,2026-06-30,0,Swiggy,565978,8901088776431,727852,TO,750.0
30151,Swiggy,MUM IM3,591495,2026-06-30,0,Swiggy,591495,8901088800754,810125,L,80.0
30150,Swiggy,GOA IM1,591495,2026-06-30,0,Swiggy,591495,8901088800754,810125,L,80.0
30160,Swiggy,NOI IM1,629254,2026-06-30,0,Swiggy,629254,8901088727952,730749,L,300.0


In [32]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [33]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])['vol_in_rum'].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-06-30,5.358
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.856
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.504
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.564
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-06-30,0.291
...,...,...,...,...,...
35071,Swiggy,VIZ IM1,810685,2026-07-31,0.008
35072,Swiggy,VIZ IM1,810685,2026-08-31,0.008
35073,Swiggy,VIZ IM1,810738,2026-06-30,17.376
35074,Swiggy,VIZ IM1,810738,2026-07-31,17.376


In [34]:
chain_fc_df = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain PSKU OTP Output/june run/QCOM Chain FC PSKU Primary_june run_04_Jun_2026.csv')
chain_fc_df

,Key2,Key,Chain,FC,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain FC PSKU P3M Val,LY Offtake Chain FC PSKU P3M Val,LY Offtake Chain FC PSKU Actuals Val,LY Offtake Chain FC PSKU Lag 1 Val,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Calculated Primary Val,Depot
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461


In [40]:
df_chk.rename(columns = {'chain_name':'Chain', 'facility_name':'FC', 'parent_material_code':'PSKU',
                         'month_date':'Month Date','vol_in_rum':'vol_in_rum_chain'}, inplace = True)
df_chk

,Chain,FC,PSKU,Month Date,vol_in_rum_chain
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-06-30,5.358
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.856
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.504
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.564
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-06-30,0.291
...,...,...,...,...,...
35071,Swiggy,VIZ IM1,810685,2026-07-31,0.008
35072,Swiggy,VIZ IM1,810685,2026-08-31,0.008
35073,Swiggy,VIZ IM1,810738,2026-06-30,17.376
35074,Swiggy,VIZ IM1,810738,2026-07-31,17.376


In [41]:
df_chk[['Chain', 'FC', 'PSKU', 'Month Date']].dtypes

Chain                 object
FC                    object
PSKU                   int64
Month Date    datetime64[ns]
dtype: object

In [ ]:
chain_fc_df[['Chain', 'FC', 'PSKU', 'Month Date']].dtypes

Chain         object
FC            object
PSKU           int64
Month Date    object
dtype: object

In [ ]:
chain_fc_df['Month Date'] = pd.to_datetime(chain_fc_df['Month Date'])
df_chk['FC'] = df_chk['FC'].str.lower()
df_chk

,Chain,FC,PSKU,Month Date,vol_in_rum_chain
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-06-30,5.358
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-07-31,5.856
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-08-31,6.504
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,6.564
4,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-06-30,0.291
...,...,...,...,...,...
35071,Swiggy,viz im1,810685,2026-07-31,0.008
35072,Swiggy,viz im1,810685,2026-08-31,0.008
35073,Swiggy,viz im1,810738,2026-06-30,17.376
35074,Swiggy,viz im1,810738,2026-07-31,17.376


In [50]:
chain_fc_df = chain_fc_df.merge(df_chk, on = ['Chain', 'FC', 'PSKU', 'Month Date'], how = 'left')
chain_fc_df

,Key2,Key,Chain,FC,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain FC PSKU P3M Val,LY Offtake Chain FC PSKU Actuals Val,LY Offtake Chain FC PSKU Lag 1 Val,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Calculated Primary Val,Depot,vol_in_rum_chain
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN


In [52]:
chain_fc_df['vol_in_rum_chain_value'] = chain_fc_df['vol_in_rum_chain']*chain_fc_df['Index Rate']/10**7
chain_fc_df

,Key2,Key,Chain,FC,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain FC PSKU Actuals Val,LY Offtake Chain FC PSKU Lag 1 Val,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Calculated Primary Val,Depot,vol_in_rum_chain,vol_in_rum_chain_value
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN


In [54]:
chain_fc_df.to_csv('chain_fc_df.csv')

In [88]:
facility_to_city_mappings_df.rename(columns = {'facility_name':'FC', 'chain':'Chain'}, inplace = True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()
facility_to_city_mappings_df

,Chain,FC,city,customer,marico_depot,status,depot_name,depot_code
0,Blinkit,farukhnagar f2 - feeder warehouse,gurugram,16012,D115,Active,NaN,D115
1,Blinkit,ahmedabad a2 - feeder warehouse,ahmedabad,15616,D354,Active,NaN,D354
2,Blinkit,hyderabad h3 - feeder warehouse,hyderabad,16005,D530,Active,NaN,D530
3,Blinkit,lucknow l5 - feeder warehouse,lucknow,15855,D113,Active,NaN,D113
4,Blinkit,super store hyderabad h2 - warehouse,hyderabad,9895,D530,Active,NaN,D530
...,...,...,...,...,...,...,...,...
144,Zepto,lko-dry-mh-sohramau,lucknow,16804,D113,Active,NaN,D113
145,Blinkit,hot mumbai m12 - feeder,mumbai,18603,D356,Active,NaN,D356
146,Blinkit,hot patna p2 - feeder,patna,18604,D233,Active,NaN,D233
147,Swiggy,scootsy logistics private limited- coimbatore,coimbatore,18366,D676,Active,NaN,D676


In [92]:
x = facility_to_city_mappings_df.copy()

In [94]:
facility_to_city_mappings_df[facility_to_city_mappings_df.duplicated(subset = ['Chain','FC'],keep=False)]
facility_to_city_mappings_df = facility_to_city_mappings_df[['Chain','FC','depot_code']].drop_duplicates()

In [96]:
chain_fc_df = chain_fc_df.merge(facility_to_city_mappings_df, on = ['Chain','FC'], how = 'left')
chain_fc_df.isnull().sum()

Key2                           0
Key                            0
Chain                          0
FC                          9580
PSKU                           0
                           ...  
Calculated Primary Val       190
Depot                       9580
vol_in_rum_chain          374117
vol_in_rum_chain_value    374117
depot_code                  9580
Length: 131, dtype: int64

In [99]:
chain_fc_df = chain_fc_df.dropna(subset = 'depot_code')
chain_fc_df

,Key2,Key,Chain,FC,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain FC PSKU Lag 1 Val,LY Offtake Chain FC PSKU Lag 2 Val,LY Offtake Chain FC PSKU Lag 3 Val,LY Offtake Chain FC PSKU Lead 1 Val,LY Offtake Chain FC PSKU Lead 2 Val,Calculated Primary Val,Depot,vol_in_rum_chain,vol_in_rum_chain_value,depot_code
0,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN,D354
1,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN,D354
2,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN,D354
3,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN,D354
4,Blinkit_ahmedabad a2 - feeder warehouse_718287,Blinkit_718287,Blinkit,ahmedabad a2 - feeder warehouse,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,D354,NaN,NaN,D354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404425,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN,D461
404426,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN,D461
404427,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN,D461
404428,Zepto_pun-dry-mh2-koregaon_811279,Zepto_811279,Zepto,pun-dry-mh2-koregaon,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,D461,NaN,NaN,D461


In [100]:
chain_depot_psku_df = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain Depot PSKU Primary_Live run_03_Jun_2026.csv')
chain_depot_psku_df

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,forecast_error,abs_forecast_error
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.001420,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.001420,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220285,zepto_nan_810628,zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,1226.374229,Skin Care,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
220286,zepto_nan_810628,zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,1226.374229,Skin Care,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
220287,zepto_nan_810628,zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,1226.374229,Skin Care,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
220288,zepto_nan_810628,zepto_nan_810628,Zepto,NaN,810628,KAYA_ML,1226.374229,Skin Care,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [104]:
chain_depot_psku_df[chain_depot_psku_df['Depot'].isna()]['Calculated Primary Val'].sum()
chain_depot_psku_df.dropna(subset = ['Depot'], inplace = True)
chain_depot_psku_df

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,forecast_error,abs_forecast_error
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-05-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-06-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216165,zepto_d674_811279,zepto_d674_811279,Zepto,d674,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
216166,zepto_d674_811279,zepto_d674_811279,Zepto,d674,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
216167,zepto_d674_811279,zepto_d674_811279,Zepto,d674,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
216168,zepto_d674_811279,zepto_d674_811279,Zepto,d674,811279,SAF_CDPRS,260000.00000,Saffola Oils,2026-06-30,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [105]:
chain_depot_psku_df.rename(columns = {'Calculated Primary Val':'Calculated Primary Val_new_model',
                                      'Calculated Primary Vol':'Calculated Primary Vol_new_model'},inplace = True)

In [110]:
chain_fc_grouped = chain_fc_df.groupby(['Chain','depot_code','PSKU','Month Date'])[['vol_in_rum_chain','vol_in_rum_chain_value',
                                                                 'Calculated Primary Val', 'Calculated Primary Vol']].sum().reset_index()

chain_fc_grouped.rename(columns = {'Calculated Primary Val':'Calculated Primary Val_old_model',
                                      'Calculated Primary Vol':'Calculated Primary Vol_old_model',
                                      'vol_in_rum_chain':'Primary_chain_vol',
                                      'vol_in_rum_chain_value':'Primary_chain_value',
                                      'depot_code':'Depot'}, inplace = True)
chain_fc_grouped['Depot'] = chain_fc_grouped['Depot'].str.lower()
chain_fc_grouped

,Chain,Depot,PSKU,Month Date,Primary_chain_vol,Primary_chain_value,Calculated Primary Val_old_model,Calculated Primary Vol_old_model
0,Blinkit,d112,718287,2026-05-31,0.0,0.0,0.0,0.0
1,Blinkit,d112,718287,2026-06-30,0.0,0.0,0.0,0.0
2,Blinkit,d112,718287,2026-07-31,0.0,0.0,0.0,0.0
3,Blinkit,d112,718287,2026-08-31,0.0,0.0,0.0,0.0
4,Blinkit,d112,718287,2026-09-30,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
214595,Zepto,d674,811279,2026-10-31,0.0,0.0,0.0,0.0
214596,Zepto,d674,811279,2026-11-30,0.0,0.0,0.0,0.0
214597,Zepto,d674,811279,2026-12-31,0.0,0.0,0.0,0.0
214598,Zepto,d674,811279,2027-01-31,0.0,0.0,0.0,0.0


In [111]:
chain_fc_grouped.columns

Index(['Chain', 'Depot', 'PSKU', 'Month Date', 'Primary_chain_vol',
       'Primary_chain_value', 'Calculated Primary Val_old_model',
       'Calculated Primary Vol_old_model'],
      dtype='object')

In [115]:
chain_depot_psku_df['Month Date'] = pd.to_datetime(chain_depot_psku_df['Month Date'])

In [117]:
chain_depot_psku_df = chain_depot_psku_df.merge(chain_fc_grouped, on = ['Chain', 'Depot', 'PSKU', 'Month Date'], how = 'left')

In [122]:
chain_depot_psku_df.to_csv('Qcom_chain_depot_psku_all_comparisons.csv')

In [121]:
chain_depot_psku_df['Primary_chain_value'].sum()

102.18353110707523

In [107]:
chain_fc_df[(chain_fc_df['Month Date'] == '2026-07-31') ]['vol_in_rum_chain_value'].sum()

27.251454272378222

In [1052]:
last_date_soh.groupby(['as_on_date'])['current_soh'].sum().reset_index()

,as_on_date,current_soh
0,2025-07-31,66300.094163
1,2025-08-31,58863.842909
2,2025-09-30,49363.157767
3,2025-10-31,72771.708935
4,2025-11-30,70224.268203
5,2025-12-31,96184.463361
6,2026-01-31,73347.849002
7,2026-02-28,77796.414711
8,2026-03-31,71152.461181
9,2026-04-30,66267.136153


In [ ]:
query = """SELECT 
        * FROM
        dev_db.data_science.trn_soh_dc_master"""
dcm_table = pd.read_sql(
    query,
    dev_conn
)
dcm_table

import pandas as pd

# Build the DataFrame with Marico Depot split into code & city
rows = [
    {"CHAIN": "Grofers", "distributor_code": 16648, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "85,75,980.43", "FC": "Kolkata K6 - Feeder Warehouse", "marico_depot_code": "D231", "marico_depot_city": "Kolkata 1"},
    {"CHAIN": "Grofers", "distributor_code": 18377, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "9,99,319.43",  "FC": "Bengaluru B5 - Feeder Warehouse", "marico_depot_code": "D673", "marico_depot_city": "Bangalore"},
    {"CHAIN": "Grofers", "distributor_code": 16950, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "3,17,265.52",  "FC": "Faridabad - Feeder Warehouse", "marico_depot_code": "D115", "marico_depot_city": "Sonipat"},
    {"CHAIN": "Swiggy",  "distributor_code": 16914, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "32,60,801.69", "FC": "AMD IM3", "marico_depot_code": "D356", "marico_depot_city": "Bhiwandi II"},
    {"CHAIN": "Swiggy",  "distributor_code": 18466, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "21,01,460.70", "FC": "HYD IM4", "marico_depot_code": "D530", "marico_depot_city": "HYDERABAD DEPOT"},
    {"CHAIN": "Swiggy",  "distributor_code": 16994, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "15,84,475.14", "FC": "NAG IM1", "marico_depot_code": "D356", "marico_depot_city": "Bhiwandi II"},
    {"CHAIN": "Zepto",   "distributor_code": 16808, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "1,70,56,520.12", "FC": "BLR-DRY-MH-Sumadhura2", "marico_depot_code": "D673", "marico_depot_city": "Bangalore"},
    # Last row had FC, then depot code (D113) on next line, then city (Lucknow) on the following line
    {"CHAIN": "Zepto",   "distributor_code": 16804, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "1,35,66,121.70", "FC": "LKO-DRY-MH-SOHRAMAU", "marico_depot_code": "D113", "marico_depot_city": "Lucknow"},
]

df = pd.DataFrame(rows)

# Clean Indian-formatted numbers -> float
num_col = "sec_ind_bpm_mth (mesr-annual_ind_rate)"
df[num_col] = df[num_col].str.replace(",", "", regex=False).astype(float)

print(df)
print("\nData types:\n", df.dtypes)
dcm_table.columns
#df.drop(columns=['sec_ind_bpm_mth (mesr-annual_ind_rate)'], inplace=True)
df.rename(columns={'distributor_code': 'customer','FC':'facility_name', 'marico_depot_city':'city',
                   'marico_depot_code':'marico_depot','CHAIN':'channel'}, inplace=True)
df

df['status'] = 'None'
df['depot_name'] = 'None'
df
df.columns = df.columns.str.upper()
df
dcm_table['channel'].unique()
dcm_table.columns = dcm_table.columns.str.lower()
dcm_table[dcm_table['customer'].isin(df['distributor_code'].unique())]
df['CHANNEL'].unique()
df["CHANNEL"] = df["CHANNEL"].replace("Grofers", "Blinkit")
df
dcm_table.columns = dcm_table.columns.str.upper()
df['STATUS'] = 'Active'
df2 = pd.concat([dcm_table, df], ignore_index=True)
df2

df2.to_csv('trn_soh_dc_master.csv', index=False)
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, df2, 
            table_name = "TRN_SOH_DC_MASTER",
            auto_create_table=True,
            overwrite = False,
)

In [1443]:
df.to_csv('city____.csv')